# Sparse Retrieval — BM25 & TF-IDF

**Goal: maximise Recall@100** — every relevant document missed here is permanently lost
to the hybrid pipeline that follows.

| Section | Models |
|---------|--------|
| §1 TF-IDF | Unigram/Bigram/Trigram on TA and FT |
| §2 BM25 variants | BM25, BM25F, BM25+, BM25L on TA/FT |
| §3 Comparison | Side-by-side Recall@100 across all models |
| §4 Domain routing | Per-domain best model → routed submission |

Results are cached to `submissions/` so re-running is instant.  
The cached `.json` + `.npy` files are consumed directly by `hybrid_pipeline.ipynb`.

---
## Imports & constants

In [37]:
import json
import math
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm

DATA_DIR        = Path('../data')
SUBMISSIONS_DIR = Path('../submissions')
SUBMISSIONS_DIR.mkdir(exist_ok=True)

TOP_K = 100

# ── Held-out mode ─────────────────────────────────────────────────────────
# Set HAS_QRELS = False when running on held-out queries (no ground truth).
# Also set QUERIES_PATH and SPECTER_QUERY_EMB to the new query files.
HAS_QRELS          = True           # ← set False for held-out
QUERIES_PATH       = DATA_DIR / 'expanded_queries.parquet'   # ← change for held-out
FORMER_QUERIES_PATH       = DATA_DIR / 'queries_1.parquet'   # ← change for held-out
QRELS_PATH         = DATA_DIR / 'qrels_1.json'         # ← ignored when HAS_QRELS=False
SPECTER_QUERY_EMB  = 'queries_1_embeddings.npy'        # ← change for held-out
# Cache prefix — change to e.g. "heldout_" to avoid overwriting training caches
CACHE_PREFIX       = ''             # ← set e.g. "heldout_" for held-out

print('Imports ready.')
print(f'HAS_QRELS={HAS_QRELS}  QUERIES_PATH={QUERIES_PATH.name}  CACHE_PREFIX={repr(CACHE_PREFIX)}')

Imports ready.
HAS_QRELS=True  QUERIES_PATH=expanded_queries.parquet  CACHE_PREFIX=''


---
## Helpers & model classes

In [38]:
# ── Data loaders ────────────────────────────────────────────
def load_queries(path): return pd.read_parquet(path)
def load_corpus(path):  return pd.read_parquet(path)
def load_qrels(path):
    with open(path) as f: return json.load(f)

# ── Text helpers ──────────────────────────────────────────
def format_text(row):
    t = str(row.get('title','') or '').strip()
    a = str(row.get('abstract','') or '').strip()
    return (t + ' ' + a).strip() if (t and a) else (t or a)

def format_text_expanded(row):
    t = str(row.get('expanded_text','') or '').strip()
    a = str(row.get('abstract','') or '').strip()
    return (t + ' ' + a).strip() if (t and a) else (t or a)

def get_ta(row):
    return str(row.get('ta','') or '').strip()

def get_body_chunks(row, min_chars=100):
    import json as _json
    meta = _json.loads(row['chunk_meta']) if isinstance(row['chunk_meta'], str) else row['chunk_meta']
    ft   = row['full_text']
    out  = []
    for i, e in enumerate(meta):
        if e['type'] == 'ta': continue
        end  = meta[i+1]['char_start'] if i+1 < len(meta) else len(ft)
        text = ft[e['char_start']:end].strip()
        if len(text) >= min_chars:
            out.append(text)
    return out

# ── TF-IDF builder ─────────────────────────────────────────
def build_tfidf(corpus_texts, query_texts, ngram_range=(1,2),
                min_df=2, max_df=0.95, max_features=None):
    vec   = TfidfVectorizer(sublinear_tf=True, min_df=min_df, max_df=max_df,
                            ngram_range=ngram_range, stop_words='english',
                            max_features=max_features)
    c_mat = vec.fit_transform(corpus_texts)
    q_mat = vec.transform(query_texts)
    return c_mat, q_mat, vec

# ── Submission builder ────────────────────────────────────
def make_submission(score_matrix, q_ids, c_ids, top_k=TOP_K):
    return {q_ids[i]: [c_ids[j] for j in np.argsort(-score_matrix[i])[:top_k]]
            for i in range(len(q_ids))}

# ── Metric helpers ─────────────────────────────────────────
def recall_at_k(ranked, relevant, k):
    if not relevant: return 0.0
    return sum(1 for d in ranked[:k] if d in relevant) / len(relevant)

def precision_at_k(ranked, relevant, k):
    if k == 0: return 0.0
    return sum(1 for d in ranked[:k] if d in relevant) / k

def mrr_at_k(ranked, relevant, k):
    for rank, doc in enumerate(ranked[:k], 1):
        if doc in relevant: return 1.0 / rank
    return 0.0

def ndcg_at_k(ranked, relevant, k):
    dcg  = sum(1/math.log2(r+1) for r,d in enumerate(ranked[:k],1) if d in relevant)
    idcg = sum(1/math.log2(r+1) for r in range(1, min(len(relevant),k)+1))
    return dcg/idcg if idcg else 0.0

def average_precision(ranked, relevant):
    if not relevant: return 0.0
    hits = score = 0
    for rank, doc in enumerate(ranked, 1):
        if doc in relevant:
            hits += 1; score += hits/rank
    return score / len(relevant)

# ── Per-query & aggregate evaluate ────────────────────────────────
def evaluate(submission, qrels, ks=None, query_domains=None, verbose=True):
    if ks is None: ks = [10, 50, 100]
    per_query = {}
    for qid, rel_list in qrels.items():
        relevant = set(rel_list)
        ranked   = submission.get(qid, [])
        q = {}
        for k in ks:
            q[f'Recall@{k}']    = recall_at_k(ranked, relevant, k)
            q[f'Precision@{k}'] = precision_at_k(ranked, relevant, k)
            q[f'MRR@{k}']       = mrr_at_k(ranked, relevant, k)
            q[f'NDCG@{k}']      = ndcg_at_k(ranked, relevant, k)
        q['AP'] = average_precision(ranked, relevant)
        per_query[qid] = q

    metric_keys = list(next(iter(per_query.values())).keys()) if per_query else []
    overall = {key: float(np.mean([per_query[q][key] for q in per_query])) for key in metric_keys}
    overall['MAP'] = overall.pop('AP', 0.0)
    overall['num_queries'] = len(per_query)
    result = {'overall': overall, 'per_query': per_query}

    if query_domains:
        per_domain = {}
        for domain in sorted(set(query_domains.values())):
            dqids = [q for q in per_query if query_domains.get(q) == domain]
            if not dqids: continue
            dm = {k: float(np.mean([per_query[q][k] for q in dqids])) for k in metric_keys}
            dm['MAP'] = dm.pop('AP', 0.0)
            dm['num_queries'] = len(dqids)
            per_domain[domain] = dm
        result['per_domain'] = per_domain

    if verbose:
        _print_results(result, ks)
    return result

def _print_results(results, ks):
    o = results['overall']
    print('\n' + '='*68 + '\nOVERALL RESULTS\n' + '='*68)
    for label, keys in [('Recall',    [f'Recall@{k}'    for k in ks]),
                        ('Precision', [f'Precision@{k}' for k in ks]),
                        ('MRR',       [f'MRR@{k}'       for k in ks]),
                        ('NDCG',      [f'NDCG@{k}'      for k in ks])]:
        print(f'{label:<14}' + ''.join(f'  @{k:>3}: {o.get(f,0):.4f}' for k,f in zip(ks,keys)))
    print(f"{'MAP':<14}  {o.get('MAP',0):.4f}")
    print(f"{'Queries':<14}  {int(o.get('num_queries',0))}")
    if 'per_domain' in results:
        print(f'\n{"-"*68}\nPER-DOMAIN  (Recall@100)\n{"-"*68}')
        print(f"  {'Domain':<28} {'Recall@100':<12} {'NDCG@10':<10} MAP")
        for domain, dm in sorted(results['per_domain'].items()):
            print(f"  {domain:<28} {dm.get('Recall@100',0):.4f}       "
                  f"{dm.get('NDCG@10',0):.4f}     {dm.get('MAP',0):.4f}")
    print()

# ── Side-by-side comparison ───────────────────────────────────────
def compare_submissions(submissions, qrels, ks=None, query_domains=None):
    if ks is None: ks = [10, 50, 100]
    names   = list(submissions.keys())
    results = {n: evaluate(s, qrels, ks=ks, query_domains=query_domains, verbose=False)
               for n, s in submissions.items()}
    overall = {n: results[n]['overall'] for n in names}
    metric_keys = [k for k in next(iter(overall.values())) if k != 'num_queries']

    col_w  = max(len(n) for n in names) + 2
    header = f"{'Metric':<20}" + ''.join(f'{n:>{col_w}}' for n in names) + '  Best'
    if len(names) == 2: header += f"  {'Delta':>8}"
    print('\n' + '='*len(header) + '\nSUBMISSION COMPARISON\n' + '='*len(header))
    print(header); print('-'*len(header))
    for metric in metric_keys:
        vals = {n: round(overall[n].get(metric,0.0),4) for n in names}
        best = max(vals, key=vals.get)
        line = f'{metric:<20}' + ''.join(f'{vals[n]:>{col_w}.4f}' for n in names) + f'  {best}'
        if len(names) == 2:
            a,b = names
            line += f'  {vals[b]-vals[a]:>+8.4f}'
        print(line)
    print()

    if query_domains:
        key    = 'Recall@100' if 'Recall@100' in metric_keys else f'Recall@{ks[-1]}'
        col_w2 = max(len(n) for n in names) + 2
        hdr2   = f"  {'Domain':<28}" + ''.join(f'{n:>{col_w2}}' for n in names) + '  Best'
        print(f'PER-DOMAIN  {key}\n{hdr2}\n  ' + '-'*(len(hdr2)-2))
        for domain in sorted(set(query_domains.values())):
            vals = {n: results[n].get('per_domain',{}).get(domain,{}).get(key,0.0) for n in names}
            best = max(vals, key=vals.get)
            print(f'  {domain:<28}' + ''.join(f'{vals[n]:>{col_w2}.4f}' for n in names) + f'  {best}')
        print()
    return results

# ── Grid-search objective: Recall@100 from a score matrix ────────────────
def quick_recall100(scores, q_ids, c_ids, qrels, k=100):
    _c = np.array(c_ids)
    vals = []
    for i, qid in enumerate(q_ids):
        rel = set(qrels.get(qid, []))
        if not rel: continue
        top = _c[np.argsort(-scores[i])[:k]]
        vals.append(sum(1 for d in top if d in rel) / len(rel))
    return float(np.mean(vals)) if vals else 0.0

# ── Disk cache helpers ─────────────────────────────────────────
def _is_cached(name):
    n = CACHE_PREFIX + name
    return ((SUBMISSIONS_DIR / f'submission_{n}.json').exists() and
            (SUBMISSIONS_DIR / f'scores_{n}.npy').exists())

def _load_cache(name):
    n = CACHE_PREFIX + name
    with open(SUBMISSIONS_DIR / f'submission_{n}.json') as _f:
        sub = json.load(_f)
    scores = np.load(str(SUBMISSIONS_DIR / f'scores_{n}.npy'))
    return sub, scores

def _save_cache(name, sub, scores):
    n = CACHE_PREFIX + name
    with open(SUBMISSIONS_DIR / f'submission_{n}.json', 'w') as _f:
        json.dump(sub, _f)
    np.save(str(SUBMISSIONS_DIR / f'scores_{n}.npy'), scores)

# ── BM25 ──────────────────────────────────────────────────────
class BM25:
    def __init__(self, k1=1.5, b=0.75): self.k1=k1; self.b=b

    def fit(self, corpus_texts):
        self._vec   = CountVectorizer(min_df=2, max_df=0.95, stop_words='english')
        tf          = self._vec.fit_transform(corpus_texts).astype(np.float32).tocsr()
        N           = tf.shape[0]
        self._dl    = np.asarray(tf.sum(axis=1)).ravel()
        self._avgdl = self._dl.mean()
        df          = np.asarray((tf>0).sum(axis=0)).ravel().astype(np.float32)
        self._idf   = np.log((N-df+0.5)/(df+0.5)+1.0)
        self._tf    = tf
        self._build_W(); return self

    def _build_W(self):
        k1,b = self.k1, self.b
        ln   = 1.0-b+b*self._dl/self._avgdl
        W    = self._tf.copy(); rows,cols = W.nonzero()
        W.data = self._idf[cols]*W.data*(k1+1.0)/(W.data+k1*ln[rows])
        self._W = W

    def score_matrix(self, query_texts):
        q = (self._vec.transform(query_texts)>0).astype(np.float32)
        return (q@self._W.T).toarray()

# ── BM25F ────────────────────────────────────────────────────
class BM25F:
    def __init__(self, k1=1.5, field_weights=None, field_b=None):
        self.k1            = k1
        self.field_weights = field_weights or {'title':3.0,'abstract':1.0}
        self.field_b       = field_b       or {'title':0.4,'abstract':0.75}

    def fit(self, field_texts):
        all_texts = [t for texts in field_texts.values() for t in texts]
        self._vec = CountVectorizer(min_df=2, max_df=0.95, stop_words='english')
        self._vec.fit(all_texts)
        self._N = len(next(iter(field_texts.values())))
        self._field_tf_raw = {f: self._vec.transform(texts).astype(np.float32).tocsr()
                               for f, texts in field_texts.items()}
        self._refit_fields(); return self

    def _refit_fields(self):
        N=self._N; ptf=None; pres=None
        for field, tf_f in self._field_tf_raw.items():
            dl    = np.asarray(tf_f.sum(axis=1)).ravel()
            avgdl = max(dl.mean(), 1.0)
            w = self.field_weights.get(field,1.0); b = self.field_b.get(field,0.75)
            ln = 1.0-b+b*dl/avgdl
            pf = tf_f.copy(); rows,_ = pf.nonzero()
            pf.data = w*pf.data/ln[rows]
            ptf  = pf  if ptf  is None else ptf+pf
            pres = (tf_f>0) if pres is None else pres+(tf_f>0)
        df = np.asarray((pres>0).sum(axis=0)).ravel().astype(np.float32)
        self._idf = np.log((N-df+0.5)/(df+0.5)+1.0)
        self._ptf = ptf.tocsr().astype(np.float32)
        self._build_W()

    def _build_W(self):
        k1=self.k1; W=self._ptf.copy(); rows,cols=W.nonzero(); pv=W.data.copy()
        W.data = self._idf[cols]*(k1+1.0)*pv/(k1+pv)
        self._W = W

    def score_matrix(self, query_texts):
        q = (self._vec.transform(query_texts)>0).astype(np.float32)
        return (q@self._W.T).toarray()

# ── BM25+ ────────────────────────────────────────────────────
class BM25Plus:
    def __init__(self, k1=1.5, b=0.75, delta=1.0): self.k1=k1; self.b=b; self.delta=delta

    def fit(self, corpus_texts):
        self._vec = CountVectorizer(min_df=3, max_df=0.95, stop_words='english')
        tf = self._vec.fit_transform(corpus_texts).astype(np.float32).tocsr()
        N  = tf.shape[0]
        self._dl    = np.asarray(tf.sum(axis=1)).ravel(); self._avgdl = self._dl.mean()
        df = np.asarray((tf>0).sum(axis=0)).ravel().astype(np.float32)
        self._idf = np.log((N-df+0.5)/(df+0.5)+1.0); self._tf = tf
        self._build_W(); return self

    def _build_W(self):
        k1,b,d = self.k1,self.b,self.delta
        ln = 1.0-b+b*self._dl/self._avgdl
        W  = self._tf.copy(); rows,cols = W.nonzero(); tv = W.data.copy()
        W.data = self._idf[cols]*(d+tv*(k1+1.0)/(tv+k1*ln[rows]))
        self._W = W

    def score_matrix(self, query_texts):
        q = (self._vec.transform(query_texts)>0).astype(np.float32)
        return (q@self._W.T).toarray()

# ── BM25L ────────────────────────────────────────────────────
class BM25L:
    def __init__(self, k1=1.5, b=0.75, delta=0.5): self.k1=k1; self.b=b; self.delta=delta

    def fit(self, corpus_texts):
        self._vec = CountVectorizer(min_df=3, max_df=0.95, stop_words='english')
        tf = self._vec.fit_transform(corpus_texts).astype(np.float32).tocsr()
        N  = tf.shape[0]
        self._dl    = np.asarray(tf.sum(axis=1)).ravel(); self._avgdl = self._dl.mean()
        df = np.asarray((tf>0).sum(axis=0)).ravel().astype(np.float32)
        self._idf = np.log((N-df+0.5)/(df+0.5)+1.0); self._tf = tf
        self._build_W(); return self

    def _build_W(self):
        k1,b,d = self.k1,self.b,self.delta
        ln = 1.0-b+b*self._dl/self._avgdl
        W  = self._tf.copy(); rows,cols = W.nonzero()
        tn = W.data/ln[rows]; td = tn+d
        W.data = self._idf[cols]*(k1+1.0)*td/(k1+td)
        self._W = W

    def score_matrix(self, query_texts):
        q = (self._vec.transform(query_texts)>0).astype(np.float32)
        return (q@self._W.T).toarray()

print('All helpers and model classes ready.')

All helpers and model classes ready.


---
## Data loading

In [39]:
queries = load_queries(QUERIES_PATH)
former_queries = load_queries(FORMER_QUERIES_PATH) if FORMER_QUERIES_PATH.exists() else None
corpus  = load_corpus(DATA_DIR / 'corpus.parquet')
qrels   = load_qrels(QRELS_PATH) if HAS_QRELS else {}

query_ids     = queries['doc_id'].tolist()
corpus_ids    = corpus['doc_id'].tolist()
query_domains = dict(zip(queries['doc_id'], queries['domain']))

# Title+Abstract — query side for all TA models
corpus_texts = [format_text(row) for _, row in tqdm(corpus.iterrows(),  total=len(corpus),  desc='corpus TA')]
query_texts  = [format_text_expanded(row) for _, row in tqdm(queries.iterrows(), total=len(queries), desc='query TA')]
query_ta     = [get_ta(row)      for _, row in tqdm(queries.iterrows(), total=len(queries), desc='query raw TA')]

# Full text — corpus only; queries always use TA
corpus_full = corpus['full_text'].fillna('').astype(str).tolist()

# Per-field texts for BM25F
title_texts    = corpus['title'].fillna('').astype(str).tolist()
abstract_texts = corpus['abstract'].fillna('').astype(str).tolist()
body_texts     = [' '.join(get_body_chunks(row, min_chars=50))
                  for _, row in tqdm(corpus.iterrows(), total=len(corpus), desc='body chunks')]

print(f'\nQueries : {len(queries)}  |  Corpus : {len(corpus)}  |  Qrels : {len(qrels)}')
print(f'Avg corpus full-text chars : {sum(len(t) for t in corpus_full)/len(corpus_full):,.0f}')

body chunks: 100%|██████████| 20000/20000 [00:09<00:00, 2165.78it/s]



Queries : 100  |  Corpus : 20000  |  Qrels : 100
Avg corpus full-text chars : 39,700


In [40]:
print(queries['expanded_text'][3])
print(former_queries['title'][3] if former_queries is not None else 'No former queries loaded.')

Parametric Acoustic Array and Its Application in Underwater Acoustic Engineering As a sound transmitting device based on the nonlinear acoustic theory, parametric acoustic array (PAA) is able to generate high directivity and low frequency broadband signals with a small aperture transducer. Due to its predominant technical advantages, PAA has been widely used in a variety of application scenarios of underwater acoustic engineering, such as sub-bottom profile measurement, underwater acoustic communication, and detection of buried targets. In this review paper, we examine some of the important advances in the PAA since it was first proposed by Westervelt in 1963. These advances include theoretical modelling for the PAA, signal processing methods, design considerations and implementation issues, and applications of the PAA in underwater acoustic engineering. Moreover, we highlight some technical challenges which impede further development of the PAA, and correspondingly give a glimpse on i

In [41]:
queries.head()

,doc_id,title,abstract,ta,full_text,chunk_meta,venue,domain,year,n_relevant,expanded_text
0,5226ac1019c028800679a3c1badccfbde9ceecef,Ecotypic Morphological and Physio-Biochemical ...,: Crop performance and yield are the results o...,Ecotypic Morphological and Physio-Biochemical ...,Ecotypic Morphological and Physio-Biochemical ...,"[{""type"": ""ta"", ""char_start"": 0, ""char_end"": 2...",Sustainability,Biology,2021,6,Ecotypic Morphological and Physio-Biochemical ...
1,2935ad9d0ce18db424cb4b72de7dd5a646a6e8e7,Virtual Sound Field of the Roman Theatre of Ma...,: In Hispania (present-day Spain and Portugal)...,Virtual Sound Field of the Roman Theatre of Ma...,Virtual Sound Field of the Roman Theatre of Ma...,"[{""type"": ""ta"", ""char_start"": 0, ""char_end"": 1...",Acoustics,History,2021,2,Virtual Sound Field of the Roman Theatre of Ma...
2,45758ead6003dac56e57d120b668ff5839bf5572,Basal Bioenergetic Abnormalities in Skeletal M...,Malignant hyperthermia (MH) and central core d...,Basal Bioenergetic Abnormalities in Skeletal M...,Basal Bioenergetic Abnormalities in Skeletal M...,"[{""type"": ""ta"", ""char_start"": 0, ""char_end"": 1...",Journal of Biological Chemistry,Biology,2010,3,Basal Bioenergetic Abnormalities in Skeletal M...
3,2e5f9523a49153361d2313eb2028b20667e3b14d,Parametric Acoustic Array and Its Application ...,As a sound transmitting device based on the no...,Parametric Acoustic Array and Its Application ...,Parametric Acoustic Array and Its Application ...,"[{""type"": ""ta"", ""char_start"": 0, ""char_end"": 1...",Italian National Conference on Sensors,Computer Science,2020,4,Parametric Acoustic Array and Its Application ...
4,11dbd657742ae4ee1863831ff8879b2967f4935b,Results of a participatory needs assessment de...,Background Drug users’ organizations have made...,Results of a participatory needs assessment de...,Results of a participatory needs assessment de...,"[{""type"": ""ta"", ""char_start"": 0, ""char_end"": 2...",Harm Reduction Journal,Psychology,2016,2,Results of a participatory needs assessment de...


---
# §1 — TF-IDF

TF-IDF with sublinear term frequency serves as a strong lexical baseline and often
provides complementary recall to BM25 (different IDF smoothing, different vocabulary
weighting).  We keep three n-gram ranges on TA and two on full text.

## §1.1 — TF-IDF on Title + Abstract

In [42]:
_ta_keys = ['tfidf_uni_ta', 'tfidf_bi_ta', 'tfidf_tri_ta']

if all(_is_cached(k) for k in _ta_keys):
    sub_tfidf_uni_ta, scores_tfidf_uni_ta = _load_cache('tfidf_uni_ta')
    sub_tfidf_bi_ta,  scores_tfidf_bi_ta  = _load_cache('tfidf_bi_ta')
    sub_tfidf_tri_ta, scores_tfidf_tri_ta = _load_cache('tfidf_tri_ta')
    print('TF-IDF TA: all variants loaded from cache. ✓')
else:
    print('Fitting TF-IDF TA unigram ...', end=' ', flush=True)
    cm_uni_ta, qm_uni_ta, _ = build_tfidf(corpus_texts, query_texts, ngram_range=(1,1))
    print(f'vocab={cm_uni_ta.shape[1]:,}')

    print('Fitting TF-IDF TA bigram  ...', end=' ', flush=True)
    cm_bi_ta, qm_bi_ta, _ = build_tfidf(corpus_texts, query_texts, ngram_range=(1,2))
    print(f'vocab={cm_bi_ta.shape[1]:,}')

    print('Fitting TF-IDF TA trigram ...', end=' ', flush=True)
    cm_tri_ta, qm_tri_ta, _ = build_tfidf(corpus_texts, query_texts, ngram_range=(1,3))
    print(f'vocab={cm_tri_ta.shape[1]:,}')

    print('\nScoring TA variants ...')
    scores_tfidf_uni_ta = cosine_similarity(qm_uni_ta, cm_uni_ta)
    scores_tfidf_bi_ta  = cosine_similarity(qm_bi_ta,  cm_bi_ta)
    scores_tfidf_tri_ta = cosine_similarity(qm_tri_ta, cm_tri_ta)

    sub_tfidf_uni_ta = make_submission(scores_tfidf_uni_ta, query_ids, corpus_ids)
    sub_tfidf_bi_ta  = make_submission(scores_tfidf_bi_ta,  query_ids, corpus_ids)
    sub_tfidf_tri_ta = make_submission(scores_tfidf_tri_ta, query_ids, corpus_ids)

    _save_cache('tfidf_uni_ta', sub_tfidf_uni_ta, scores_tfidf_uni_ta)
    _save_cache('tfidf_bi_ta',  sub_tfidf_bi_ta,  scores_tfidf_bi_ta)
    _save_cache('tfidf_tri_ta', sub_tfidf_tri_ta, scores_tfidf_tri_ta)
    print('TF-IDF TA: trained and saved. ✓')

compare_submissions(
    {'TF-IDF uni (TA)': sub_tfidf_uni_ta,
     'TF-IDF bi (TA)':  sub_tfidf_bi_ta,
     'TF-IDF tri (TA)': sub_tfidf_tri_ta},
    qrels, ks=[10, 50, 100], query_domains=query_domains,
)

Fitting TF-IDF TA unigram ... vocab=44,105
Fitting TF-IDF TA bigram  ... vocab=329,198
Fitting TF-IDF TA trigram ... vocab=406,841

Scoring TA variants ...
TF-IDF TA: trained and saved. ✓

SUBMISSION COMPARISON
Metric                TF-IDF uni (TA)   TF-IDF bi (TA)  TF-IDF tri (TA)  Best
-----------------------------------------------------------------------------
Recall@10                      0.4563           0.4510           0.4514  TF-IDF uni (TA)
Precision@10                   0.2350           0.2340           0.2340  TF-IDF uni (TA)
MRR@10                         0.6651           0.6861           0.6621  TF-IDF bi (TA)
NDCG@10                        0.4899           0.4968           0.4879  TF-IDF bi (TA)
Recall@50                      0.6543           0.6780           0.6680  TF-IDF bi (TA)
Precision@50                   0.0868           0.0874           0.0866  TF-IDF bi (TA)
MRR@50                         0.6691           0.6917           0.6678  TF-IDF bi (TA)
NDCG@50        

{'TF-IDF uni (TA)': {'overall': {'Recall@10': 0.45628704180629337,
   'Precision@10': 0.23500000000000004,
   'MRR@10': 0.6651388888888888,
   'NDCG@10': 0.48993558666679804,
   'Recall@50': 0.6542947738852405,
   'Precision@50': 0.08680000000000002,
   'MRR@50': 0.6691193580808721,
   'NDCG@50': 0.5327326027245768,
   'Recall@100': 0.7415041260686838,
   'Precision@100': 0.0524,
   'MRR@100': 0.6692642856171039,
   'NDCG@100': 0.556845904372692,
   'MAP': 0.37960497560000783,
   'num_queries': 100},
  'per_query': {'002f5c970bac90e79f2b20e2d9e155d7619d6905': {'Recall@10': 0.5,
    'Precision@10': 0.2,
    'MRR@10': 1.0,
    'NDCG@10': 0.5135312443624667,
    'Recall@50': 0.5,
    'Precision@50': 0.04,
    'MRR@50': 1.0,
    'NDCG@50': 0.5135312443624667,
    'Recall@100': 0.5,
    'Precision@100': 0.02,
    'MRR@100': 1.0,
    'NDCG@100': 0.5135312443624667,
    'AP': 0.3125},
   '01ce421a014b560cd7b71029a22fd042990cda52': {'Recall@10': 0.35714285714285715,
    'Precision@10': 0.5,
  

## §1.2 — TF-IDF on Full Text

In [44]:
_ft_keys = ['tfidf_uni_ft', 'tfidf_bi_ft']

if all(_is_cached(k) for k in _ft_keys):
    sub_tfidf_uni_ft, scores_tfidf_uni_ft = _load_cache('tfidf_uni_ft')
    sub_tfidf_bi_ft,  scores_tfidf_bi_ft  = _load_cache('tfidf_bi_ft')
    print('TF-IDF FT: both variants loaded from cache. ✓')
else:
    print('Fitting TF-IDF FT unigram ...', end=' ', flush=True)
    cm_uni_ft, qm_uni_ft, _ = build_tfidf(corpus_full, query_texts,
                                            ngram_range=(1,1), min_df=3, max_features=400_000)
    print(f'vocab={cm_uni_ft.shape[1]:,}')

    print('Fitting TF-IDF FT bigram  ...', end=' ', flush=True)
    cm_bi_ft, qm_bi_ft, _ = build_tfidf(corpus_full, query_texts,
                                           ngram_range=(1,2), min_df=3, max_features=400_000)
    print(f'vocab={cm_bi_ft.shape[1]:,}')

    print('\nScoring FT variants ...')
    scores_tfidf_uni_ft = cosine_similarity(qm_uni_ft, cm_uni_ft)
    scores_tfidf_bi_ft  = cosine_similarity(qm_bi_ft,  cm_bi_ft)

    sub_tfidf_uni_ft = make_submission(scores_tfidf_uni_ft, query_ids, corpus_ids)
    sub_tfidf_bi_ft  = make_submission(scores_tfidf_bi_ft,  query_ids, corpus_ids)

    _save_cache('tfidf_uni_ft', sub_tfidf_uni_ft, scores_tfidf_uni_ft)
    _save_cache('tfidf_bi_ft',  sub_tfidf_bi_ft,  scores_tfidf_bi_ft)
    print('TF-IDF FT: trained and saved. ✓')

compare_submissions(
    {'TF-IDF uni (FT)': sub_tfidf_uni_ft,
     'TF-IDF bi (FT)':  sub_tfidf_bi_ft},
    qrels, ks=[10, 50, 100], query_domains=query_domains,
)

TF-IDF FT: both variants loaded from cache. ✓

SUBMISSION COMPARISON
Metric                TF-IDF uni (FT)   TF-IDF bi (FT)  Best     Delta
----------------------------------------------------------------------
Recall@10                      0.5507           0.5596  TF-IDF bi (FT)   +0.0089
Precision@10                   0.2820           0.2820  TF-IDF uni (FT)   +0.0000
MRR@10                         0.6879           0.7139  TF-IDF bi (FT)   +0.0260
NDCG@10                        0.5677           0.5734  TF-IDF bi (FT)   +0.0057
Recall@50                      0.7682           0.7652  TF-IDF uni (FT)   -0.0030
Precision@50                   0.1028           0.1028  TF-IDF uni (FT)   +0.0000
MRR@50                         0.6919           0.7157  TF-IDF bi (FT)   +0.0238
NDCG@50                        0.6163           0.6162  TF-IDF uni (FT)   -0.0001
Recall@100                     0.8430           0.8305  TF-IDF uni (FT)   -0.0125
Precision@100                  0.0608           0.0602 

{'TF-IDF uni (FT)': {'overall': {'Recall@10': 0.5506869105565267,
   'Precision@10': 0.28200000000000003,
   'MRR@10': 0.6879126984126984,
   'NDCG@10': 0.5676513772599772,
   'Recall@50': 0.7682276083304452,
   'Precision@50': 0.10280000000000002,
   'MRR@50': 0.6919393859276581,
   'NDCG@50': 0.61631416449937,
   'Recall@100': 0.8429775321577604,
   'Precision@100': 0.0608,
   'MRR@100': 0.6921354643590308,
   'NDCG@100': 0.6380158571475933,
   'MAP': 0.4671905064169905,
   'num_queries': 100},
  'per_query': {'002f5c970bac90e79f2b20e2d9e155d7619d6905': {'Recall@10': 0.5,
    'Precision@10': 0.2,
    'MRR@10': 1.0,
    'NDCG@10': 0.5855700749881525,
    'Recall@50': 0.5,
    'Precision@50': 0.04,
    'MRR@50': 1.0,
    'NDCG@50': 0.5855700749881525,
    'Recall@100': 0.5,
    'Precision@100': 0.02,
    'MRR@100': 1.0,
    'NDCG@100': 0.5855700749881525,
    'AP': 0.41666666666666663},
   '01ce421a014b560cd7b71029a22fd042990cda52': {'Recall@10': 0.5,
    'Precision@10': 0.7,
    'MRR@

---
# §2 — BM25 Variants

**Grid search objective: Recall@100** — we optimise hyper-parameters to maximise
the fraction of relevant documents that appear in the top-100, not precision at the
very top.  The cross-encoder reranker will handle precision later.

All models fit the corpus vocabulary once; only the weight matrix `W` is rebuilt
during the grid search, so iteration is cheap.

## §2.1 — BM25 on Title + Abstract

Standard Okapi BM25 on the concatenated title + abstract of corpus documents.

In [45]:
if _is_cached('bm25_ta'):
    sub_bm25_ta, scores_bm25_ta = _load_cache('bm25_ta')
    print('BM25 TA: loaded from cache. ✓')
else:
    BM25_TA_GRID = {'k1': [1.2, 1.5, 1.8, 2.0], 'b': [0.5, 0.65, 0.75, 0.85]}
    print('Fitting BM25 TA (one vocab fit) ...')
    _bm25_ta = BM25().fit(corpus_texts)
    print(f'  vocab size: {len(_bm25_ta._vec.vocabulary_):,}\n')

    combos       = list(product(BM25_TA_GRID['k1'], BM25_TA_GRID['b']))
    bm25_ta_rows = []
    for k1, b in tqdm(combos, desc='BM25 TA grid'):
        _bm25_ta.k1 = k1; _bm25_ta.b = b; _bm25_ta._build_W()
        scores = _bm25_ta.score_matrix(query_texts)
        r100   = quick_recall100(scores, query_ids, corpus_ids, qrels)
        bm25_ta_rows.append((r100, k1, b))

    bm25_ta_rows.sort(reverse=True)
    print(f"\n{'k1':>6}  {'b':>6}  Recall@100"); print('-'*30)
    for r100, k1, b in bm25_ta_rows:
        print(f'{k1:>6.2f}  {b:>6.2f}  {r100:.4f}')

    best = bm25_ta_rows[0]
    _bm25_ta.k1 = best[1]; _bm25_ta.b = best[2]; _bm25_ta._build_W()
    scores_bm25_ta = _bm25_ta.score_matrix(query_texts)
    sub_bm25_ta    = make_submission(scores_bm25_ta, query_ids, corpus_ids)
    print(f'\nBest BM25 TA: k1={best[1]}, b={best[2]}  Recall@100={best[0]:.4f}')
    _save_cache('bm25_ta', sub_bm25_ta, scores_bm25_ta)
    print('BM25 TA: saved. ✓')

evaluate(sub_bm25_ta, qrels, ks=[10, 50, 100], query_domains=query_domains)

Fitting BM25 TA (one vocab fit) ...
  vocab size: 44,105



BM25 TA grid: 100%|██████████| 16/16 [00:05<00:00,  3.10it/s]



    k1       b  Recall@100
------------------------------
  1.80    0.85  0.6932
  2.00    0.85  0.6930
  1.50    0.85  0.6919
  1.80    0.75  0.6903
  2.00    0.75  0.6878
  1.50    0.75  0.6874
  1.20    0.85  0.6863
  1.50    0.65  0.6830
  2.00    0.65  0.6800
  1.80    0.65  0.6793
  1.20    0.75  0.6781
  2.00    0.50  0.6729
  1.20    0.65  0.6693
  1.80    0.50  0.6683
  1.50    0.50  0.6666
  1.20    0.50  0.6572

Best BM25 TA: k1=1.8, b=0.85  Recall@100=0.6932
BM25 TA: saved. ✓

OVERALL RESULTS
Recall          @ 10: 0.4112  @ 50: 0.6021  @100: 0.6932
Precision       @ 10: 0.2100  @ 50: 0.0790  @100: 0.0481
MRR             @ 10: 0.6271  @ 50: 0.6312  @100: 0.6312
NDCG            @ 10: 0.4449  @ 50: 0.4857  @100: 0.5102
MAP             0.3382
Queries         100

--------------------------------------------------------------------
PER-DOMAIN  (Recall@100)
--------------------------------------------------------------------
  Domain                       Recall@100   NDCG@10   

{'overall': {'Recall@10': 0.41124374787753964,
  'Precision@10': 0.21000000000000005,
  'MRR@10': 0.6270634920634921,
  'NDCG@10': 0.4449483944645688,
  'Recall@50': 0.6020852960734377,
  'Precision@50': 0.079,
  'MRR@50': 0.6312414659768804,
  'NDCG@50': 0.48567936396487915,
  'Recall@100': 0.6932157789300052,
  'Precision@100': 0.048100000000000004,
  'MRR@100': 0.6312414659768804,
  'NDCG@100': 0.5102102179950523,
  'MAP': 0.3382484732062085,
  'num_queries': 100},
 'per_query': {'002f5c970bac90e79f2b20e2d9e155d7619d6905': {'Recall@10': 0.5,
   'Precision@10': 0.2,
   'MRR@10': 1.0,
   'NDCG@10': 0.5205067333228022,
   'Recall@50': 0.5,
   'Precision@50': 0.04,
   'MRR@50': 1.0,
   'NDCG@50': 0.5205067333228022,
   'Recall@100': 0.5,
   'Precision@100': 0.02,
   'MRR@100': 1.0,
   'NDCG@100': 0.5205067333228022,
   'AP': 0.3214285714285714},
  '01ce421a014b560cd7b71029a22fd042990cda52': {'Recall@10': 0.2857142857142857,
   'Precision@10': 0.4,
   'MRR@10': 1.0,
   'NDCG@10': 0.55414

## §2.2 — BM25 on Full Text

Same BM25 formula but the corpus side uses the full document text (body sections
in addition to title + abstract).  Query side always uses the TA field.

In [46]:
if _is_cached('bm25_ft'):
    sub_bm25_ft, scores_bm25_ft = _load_cache('bm25_ft')
    print('BM25 FT: loaded from cache. ✓')
else:
    BM25_FT_GRID = {'k1': [1.2, 1.5, 1.8, 2.0], 'b': [0.5, 0.65, 0.75, 0.85]}
    print('Fitting BM25 FT (one vocab fit) ...')
    _bm25_ft = BM25().fit(corpus_full)
    print(f'  vocab size: {len(_bm25_ft._vec.vocabulary_):,}\n')

    combos       = list(product(BM25_FT_GRID['k1'], BM25_FT_GRID['b']))
    bm25_ft_rows = []
    for k1, b in tqdm(combos, desc='BM25 FT grid'):
        _bm25_ft.k1 = k1; _bm25_ft.b = b; _bm25_ft._build_W()
        scores = _bm25_ft.score_matrix(query_texts)
        r100   = quick_recall100(scores, query_ids, corpus_ids, qrels)
        bm25_ft_rows.append((r100, k1, b))

    bm25_ft_rows.sort(reverse=True)
    print(f"\n{'k1':>6}  {'b':>6}  Recall@100"); print('-'*30)
    for r100, k1, b in bm25_ft_rows:
        print(f'{k1:>6.2f}  {b:>6.2f}  {r100:.4f}')

    best = bm25_ft_rows[0]
    _bm25_ft.k1 = best[1]; _bm25_ft.b = best[2]; _bm25_ft._build_W()
    scores_bm25_ft = _bm25_ft.score_matrix(query_texts)
    sub_bm25_ft    = make_submission(scores_bm25_ft, query_ids, corpus_ids)
    print(f'\nBest BM25 FT: k1={best[1]}, b={best[2]}  Recall@100={best[0]:.4f}')
    _save_cache('bm25_ft', sub_bm25_ft, scores_bm25_ft)
    print('BM25 FT: saved. ✓')

evaluate(sub_bm25_ft, qrels, ks=[10, 50, 100], query_domains=query_domains)

Fitting BM25 FT (one vocab fit) ...
  vocab size: 237,961



BM25 FT grid: 100%|██████████| 16/16 [00:21<00:00,  1.36s/it]



    k1       b  Recall@100
------------------------------
  2.00    0.85  0.7917
  1.80    0.85  0.7872
  2.00    0.75  0.7864
  1.50    0.85  0.7832
  1.80    0.75  0.7806
  2.00    0.65  0.7778
  1.50    0.75  0.7757
  1.80    0.65  0.7721
  1.20    0.85  0.7720
  2.00    0.50  0.7704
  1.20    0.75  0.7701
  1.50    0.65  0.7672
  1.80    0.50  0.7667
  1.20    0.65  0.7649
  1.50    0.50  0.7622
  1.20    0.50  0.7539

Best BM25 FT: k1=2.0, b=0.85  Recall@100=0.7917
BM25 FT: saved. ✓

OVERALL RESULTS
Recall          @ 10: 0.4708  @ 50: 0.7289  @100: 0.7917
Precision       @ 10: 0.2430  @ 50: 0.0968  @100: 0.0562
MRR             @ 10: 0.6943  @ 50: 0.7021  @100: 0.7023
NDCG            @ 10: 0.5217  @ 50: 0.5835  @100: 0.6019
MAP             0.4278
Queries         100

--------------------------------------------------------------------
PER-DOMAIN  (Recall@100)
--------------------------------------------------------------------
  Domain                       Recall@100   NDCG@10   

{'overall': {'Recall@10': 0.47084378629073476,
  'Precision@10': 0.243,
  'MRR@10': 0.6943333333333332,
  'NDCG@10': 0.5217320583643321,
  'Recall@50': 0.7289137542965668,
  'Precision@50': 0.0968,
  'MRR@50': 0.702052849683539,
  'NDCG@50': 0.5835348984832666,
  'Recall@100': 0.7917477862743638,
  'Precision@100': 0.0562,
  'MRR@100': 0.7023032337081164,
  'NDCG@100': 0.6018562778385844,
  'MAP': 0.4278249933669757,
  'num_queries': 100},
 'per_query': {'002f5c970bac90e79f2b20e2d9e155d7619d6905': {'Recall@10': 0.5,
   'Precision@10': 0.2,
   'MRR@10': 1.0,
   'NDCG@10': 0.5205067333228022,
   'Recall@50': 0.5,
   'Precision@50': 0.04,
   'MRR@50': 1.0,
   'NDCG@50': 0.5205067333228022,
   'Recall@100': 0.5,
   'Precision@100': 0.02,
   'MRR@100': 1.0,
   'NDCG@100': 0.5205067333228022,
   'AP': 0.3214285714285714},
  '01ce421a014b560cd7b71029a22fd042990cda52': {'Recall@10': 0.42857142857142855,
   'Precision@10': 0.6,
   'MRR@10': 1.0,
   'NDCG@10': 0.7116179486056782,
   'Recall@50':

## §2.3 — BM25F on Title + Abstract

BM25F applies per-field length normalisation and field weighting before computing a
single IDF-weighted score.  Title terms are boosted over abstract terms.

In [47]:
if _is_cached('bm25f_ta'):
    sub_bm25f_ta, scores_bm25f_ta = _load_cache('bm25f_ta')
    print('BM25F TA: loaded from cache. ✓')
else:
    BM25F_TA_GRID = {
        'k1':         [1.5, 2.0],
        'title_w':    [2.0, 3.0, 5.0],
        'title_b':    [0.3, 0.4, 0.5],
        'abstract_b': [0.65, 0.75, 0.85],
    }
    print('Fitting BM25F TA (one vocab fit) ...')
    _bm25f_ta = BM25F().fit({'title': title_texts, 'abstract': abstract_texts})
    print(f'  vocab size: {len(_bm25f_ta._vec.vocabulary_):,}\n')

    g = BM25F_TA_GRID
    outer         = list(product(g['title_w'], g['title_b'], g['abstract_b']))
    bm25f_ta_rows = []
    for tw, tb, ab in tqdm(outer, desc='BM25F TA grid'):
        _bm25f_ta.field_weights = {'title': tw, 'abstract': 1.0}
        _bm25f_ta.field_b       = {'title': tb, 'abstract': ab}
        _bm25f_ta._refit_fields()
        for k1 in g['k1']:
            _bm25f_ta.k1 = k1; _bm25f_ta._build_W()
            scores = _bm25f_ta.score_matrix(query_texts)
            r100   = quick_recall100(scores, query_ids, corpus_ids, qrels)
            bm25f_ta_rows.append((r100, k1, tw, tb, ab))

    bm25f_ta_rows.sort(reverse=True)
    print(f"\n{'k1':>4} {'tw':>4} {'tb':>5} {'ab':>5}  Recall@100"); print('-'*40)
    for r100, k1, tw, tb, ab in bm25f_ta_rows[:10]:
        print(f'{k1:>4.1f} {tw:>4.1f} {tb:>5.2f} {ab:>5.2f}  {r100:.4f}')

    best = bm25f_ta_rows[0]
    _bm25f_ta.k1            = best[1]
    _bm25f_ta.field_weights = {'title': best[2], 'abstract': 1.0}
    _bm25f_ta.field_b       = {'title': best[3], 'abstract': best[4]}
    _bm25f_ta._refit_fields(); _bm25f_ta._build_W()
    scores_bm25f_ta = _bm25f_ta.score_matrix(query_texts)
    sub_bm25f_ta    = make_submission(scores_bm25f_ta, query_ids, corpus_ids)
    print(f'\nBest BM25F TA: k1={best[1]}, tw={best[2]}, tb={best[3]}, ab={best[4]}  '
          f'Recall@100={best[0]:.4f}')
    _save_cache('bm25f_ta', sub_bm25f_ta, scores_bm25f_ta)
    print('BM25F TA: saved. ✓')

evaluate(sub_bm25f_ta, qrels, ks=[10, 50, 100], query_domains=query_domains)

Fitting BM25F TA (one vocab fit) ...
  vocab size: 48,985



BM25F TA grid: 100%|██████████| 27/27 [00:21<00:00,  1.26it/s]



  k1   tw    tb    ab  Recall@100
----------------------------------------
 2.0  3.0  0.30  0.85  0.7018
 2.0  3.0  0.40  0.75  0.7002
 2.0  3.0  0.30  0.75  0.7002
 2.0  3.0  0.50  0.75  0.6995
 2.0  5.0  0.40  0.85  0.6986
 2.0  5.0  0.30  0.85  0.6986
 2.0  3.0  0.40  0.85  0.6985
 2.0  5.0  0.50  0.85  0.6985
 2.0  5.0  0.50  0.75  0.6979
 2.0  3.0  0.50  0.85  0.6978

Best BM25F TA: k1=2.0, tw=3.0, tb=0.3, ab=0.85  Recall@100=0.7018
BM25F TA: saved. ✓

OVERALL RESULTS
Recall          @ 10: 0.4237  @ 50: 0.6095  @100: 0.7018
Precision       @ 10: 0.2150  @ 50: 0.0802  @100: 0.0488
MRR             @ 10: 0.6324  @ 50: 0.6353  @100: 0.6353
NDCG            @ 10: 0.4551  @ 50: 0.4932  @100: 0.5179
MAP             0.3449
Queries         100

--------------------------------------------------------------------
PER-DOMAIN  (Recall@100)
--------------------------------------------------------------------
  Domain                       Recall@100   NDCG@10    MAP
  Art                      

{'overall': {'Recall@10': 0.4236907175745093,
  'Precision@10': 0.21500000000000002,
  'MRR@10': 0.6324206349206349,
  'NDCG@10': 0.4550598586147331,
  'Recall@50': 0.6094754209758577,
  'Precision@50': 0.08020000000000001,
  'MRR@50': 0.6352864049893462,
  'NDCG@50': 0.49324990389284884,
  'Recall@100': 0.701821369669341,
  'Precision@100': 0.04880000000000001,
  'MRR@100': 0.6352864049893462,
  'NDCG@100': 0.5179237642452152,
  'MAP': 0.34490914458789645,
  'num_queries': 100},
 'per_query': {'002f5c970bac90e79f2b20e2d9e155d7619d6905': {'Recall@10': 0.5,
   'Precision@10': 0.2,
   'MRR@10': 1.0,
   'NDCG@10': 0.5413996682199069,
   'Recall@50': 0.5,
   'Precision@50': 0.04,
   'MRR@50': 1.0,
   'NDCG@50': 0.5413996682199069,
   'Recall@100': 0.5,
   'Precision@100': 0.02,
   'MRR@100': 1.0,
   'NDCG@100': 0.5413996682199069,
   'AP': 0.35},
  '01ce421a014b560cd7b71029a22fd042990cda52': {'Recall@10': 0.2857142857142857,
   'Precision@10': 0.4,
   'MRR@10': 1.0,
   'NDCG@10': 0.5541432

## §2.4 — BM25F on Full Text (title + abstract + body)

Three-field BM25F: title, abstract, body sections.  Each field gets its own length
normalisation `b` and a weight multiplier.  Body is the normalisation anchor (weight=1).

In [48]:
if _is_cached('bm25f_ft'):
    sub_bm25f_ft, scores_bm25f_ft = _load_cache('bm25f_ft')
    print('BM25F FT: loaded from cache. ✓')
else:
    BM25F_FT_GRID = {
        'k1':         [1.5, 2.0],
        'title_w':    [3.0, 5.0],
        'abstract_w': [1.0, 2.0],
        'title_b':    [0.3, 0.5],
        'abstract_b': [0.65, 0.85],
        'body_b':     [0.75, 0.9],
    }
    print('Fitting BM25F FT (one vocab fit) ...')
    _bm25f_ft = BM25F().fit({'title': title_texts, 'abstract': abstract_texts, 'body': body_texts})
    print(f'  vocab size: {len(_bm25f_ft._vec.vocabulary_):,}\n')

    g = BM25F_FT_GRID
    outer         = list(product(g['title_w'], g['abstract_w'],
                                  g['title_b'], g['abstract_b'], g['body_b']))
    bm25f_ft_rows = []
    for tw, aw, tb, ab, bb in tqdm(outer, desc='BM25F FT grid'):
        _bm25f_ft.field_weights = {'title': tw, 'abstract': aw, 'body': 1.0}
        _bm25f_ft.field_b       = {'title': tb, 'abstract': ab, 'body': bb}
        _bm25f_ft._refit_fields()
        for k1 in g['k1']:
            _bm25f_ft.k1 = k1; _bm25f_ft._build_W()
            scores = _bm25f_ft.score_matrix(query_texts)
            r100   = quick_recall100(scores, query_ids, corpus_ids, qrels)
            bm25f_ft_rows.append((r100, k1, tw, aw, tb, ab, bb))

    bm25f_ft_rows.sort(reverse=True)
    print(f"\n{'k1':>4} {'tw':>4} {'aw':>4} {'tb':>5} {'ab':>5} {'bb':>5}  Recall@100")
    print('-'*54)
    for row in bm25f_ft_rows[:10]:
        r100, k1, tw, aw, tb, ab, bb = row
        print(f'{k1:>4.1f} {tw:>4.1f} {aw:>4.1f} {tb:>5.2f} {ab:>5.2f} {bb:>5.2f}  {r100:.4f}')

    best = bm25f_ft_rows[0]
    _bm25f_ft.k1            = best[1]
    _bm25f_ft.field_weights = {'title': best[2], 'abstract': best[3], 'body': 1.0}
    _bm25f_ft.field_b       = {'title': best[4], 'abstract': best[5], 'body': best[6]}
    _bm25f_ft._refit_fields(); _bm25f_ft._build_W()
    scores_bm25f_ft = _bm25f_ft.score_matrix(query_texts)
    sub_bm25f_ft    = make_submission(scores_bm25f_ft, query_ids, corpus_ids)
    print(f'\nBest BM25F FT: k1={best[1]}, tw={best[2]}, aw={best[3]}, '
          f'tb={best[4]}, ab={best[5]}, bb={best[6]}  Recall@100={best[0]:.4f}')
    _save_cache('bm25f_ft', sub_bm25f_ft, scores_bm25f_ft)
    print('BM25F FT: saved. ✓')

evaluate(sub_bm25f_ft, qrels, ks=[10, 50, 100], query_domains=query_domains)

Fitting BM25F FT (one vocab fit) ...
  vocab size: 229,270



BM25F FT grid: 100%|██████████| 32/32 [02:24<00:00,  4.51s/it]



  k1   tw   aw    tb    ab    bb  Recall@100
------------------------------------------------------
 2.0  5.0  1.0  0.30  0.65  0.90  0.7677
 2.0  5.0  1.0  0.50  0.65  0.90  0.7627
 2.0  5.0  2.0  0.50  0.85  0.90  0.7617
 2.0  5.0  2.0  0.50  0.65  0.90  0.7617
 2.0  5.0  2.0  0.30  0.85  0.90  0.7617
 2.0  5.0  2.0  0.30  0.65  0.90  0.7617
 2.0  3.0  1.0  0.50  0.85  0.90  0.7608
 2.0  3.0  1.0  0.50  0.65  0.90  0.7608
 2.0  3.0  1.0  0.30  0.85  0.90  0.7608
 2.0  3.0  1.0  0.30  0.65  0.90  0.7608

Best BM25F FT: k1=2.0, tw=5.0, aw=1.0, tb=0.3, ab=0.65, bb=0.9  Recall@100=0.7677
BM25F FT: saved. ✓

OVERALL RESULTS
Recall          @ 10: 0.4519  @ 50: 0.6811  @100: 0.7677
Precision       @ 10: 0.2290  @ 50: 0.0910  @100: 0.0535
MRR             @ 10: 0.6574  @ 50: 0.6618  @100: 0.6624
NDCG            @ 10: 0.4868  @ 50: 0.5399  @100: 0.5628
MAP             0.3857
Queries         100

--------------------------------------------------------------------
PER-DOMAIN  (Recall@100)
----

{'overall': {'Recall@10': 0.4519117359224763,
  'Precision@10': 0.22900000000000006,
  'MRR@10': 0.6574404761904762,
  'NDCG@10': 0.48681069132389637,
  'Recall@50': 0.6810993731506143,
  'Precision@50': 0.091,
  'MRR@50': 0.6618337551163382,
  'NDCG@50': 0.5398528046837189,
  'Recall@100': 0.7677297681821535,
  'Precision@100': 0.0535,
  'MRR@100': 0.6624067801977327,
  'NDCG@100': 0.5628434488716568,
  'MAP': 0.38566337966808634,
  'num_queries': 100},
 'per_query': {'002f5c970bac90e79f2b20e2d9e155d7619d6905': {'Recall@10': 0.25,
   'Precision@10': 0.1,
   'MRR@10': 1.0,
   'NDCG@10': 0.3903800499921017,
   'Recall@50': 0.5,
   'Precision@50': 0.04,
   'MRR@50': 1.0,
   'NDCG@50': 0.49291318861032357,
   'Recall@100': 0.5,
   'Precision@100': 0.02,
   'MRR@100': 1.0,
   'NDCG@100': 0.49291318861032357,
   'AP': 0.28846153846153844},
  '01ce421a014b560cd7b71029a22fd042990cda52': {'Recall@10': 0.35714285714285715,
   'Precision@10': 0.5,
   'MRR@10': 1.0,
   'NDCG@10': 0.63715237978958

## §2.5 — BM25+ on Full Text

BM25+ adds a floor `delta` to the within-document term frequency component, preventing
long documents from being penalised too heavily for terms that appear rarely.

In [49]:
if _is_cached('bm25plus_ft'):
    sub_bm25p_ft, scores_bm25p_ft = _load_cache('bm25plus_ft')
    print('BM25+ FT: loaded from cache. ✓')
else:
    BM25P_GRID = {'k1': [1.5, 2.0, 2.5], 'b': [0.65, 0.75, 0.85, 0.9], 'delta': [0.5, 1.0, 1.5]}
    print('Fitting BM25+ (one vocab fit) ...')
    _bm25p = BM25Plus().fit(corpus_full)
    print(f'  vocab size: {len(_bm25p._vec.vocabulary_):,}\n')

    combos     = list(product(BM25P_GRID['k1'], BM25P_GRID['b'], BM25P_GRID['delta']))
    bm25p_rows = []
    for k1, b, delta in tqdm(combos, desc='BM25+ grid'):
        _bm25p.k1 = k1; _bm25p.b = b; _bm25p.delta = delta; _bm25p._build_W()
        scores = _bm25p.score_matrix(query_texts)
        r100   = quick_recall100(scores, query_ids, corpus_ids, qrels)
        bm25p_rows.append((r100, k1, b, delta))

    bm25p_rows.sort(reverse=True)
    print(f"\n{'k1':>5} {'b':>5} {'delta':>6}  Recall@100"); print('-'*34)
    for r100, k1, b, delta in bm25p_rows[:10]:
        print(f'{k1:>5.1f} {b:>5.2f} {delta:>6.1f}  {r100:.4f}')

    best = bm25p_rows[0]
    _bm25p.k1 = best[1]; _bm25p.b = best[2]; _bm25p.delta = best[3]; _bm25p._build_W()
    scores_bm25p_ft = _bm25p.score_matrix(query_texts)
    sub_bm25p_ft    = make_submission(scores_bm25p_ft, query_ids, corpus_ids)
    print(f'\nBest BM25+: k1={best[1]}, b={best[2]}, delta={best[3]}  Recall@100={best[0]:.4f}')
    _save_cache('bm25plus_ft', sub_bm25p_ft, scores_bm25p_ft)
    print('BM25+ FT: saved. ✓')

evaluate(sub_bm25p_ft, qrels, ks=[10, 50, 100], query_domains=query_domains)

Fitting BM25+ (one vocab fit) ...
  vocab size: 162,542



BM25+ grid: 100%|██████████| 36/36 [00:27<00:00,  1.31it/s]



   k1     b  delta  Recall@100
----------------------------------
  2.5  0.90    0.5  0.7841
  2.5  0.85    0.5  0.7828
  2.0  0.90    0.5  0.7777
  2.5  0.75    0.5  0.7758
  2.5  0.90    1.0  0.7751
  2.0  0.85    0.5  0.7749
  2.5  0.85    1.0  0.7739
  2.5  0.65    0.5  0.7707
  1.5  0.90    0.5  0.7702
  2.0  0.90    1.0  0.7686

Best BM25+: k1=2.5, b=0.9, delta=0.5  Recall@100=0.7841
BM25+ FT: saved. ✓

OVERALL RESULTS
Recall          @ 10: 0.4742  @ 50: 0.7226  @100: 0.7841
Precision       @ 10: 0.2430  @ 50: 0.0958  @100: 0.0555
MRR             @ 10: 0.6773  @ 50: 0.6836  @100: 0.6838
NDCG            @ 10: 0.5141  @ 50: 0.5723  @100: 0.5903
MAP             0.4171
Queries         100

--------------------------------------------------------------------
PER-DOMAIN  (Recall@100)
--------------------------------------------------------------------
  Domain                       Recall@100   NDCG@10    MAP
  Art                          0.5000       0.5205     0.3214
  Biology     

{'overall': {'Recall@10': 0.47415532217269324,
  'Precision@10': 0.243,
  'MRR@10': 0.6772738095238096,
  'NDCG@10': 0.5141243453198979,
  'Recall@50': 0.7226172707254931,
  'Precision@50': 0.0958,
  'MRR@50': 0.6836279985002601,
  'NDCG@50': 0.5723277081843914,
  'Recall@100': 0.7840930802605028,
  'Precision@100': 0.05550000000000001,
  'MRR@100': 0.6837613318335934,
  'NDCG@100': 0.590303467087585,
  'MAP': 0.4170574721507431,
  'num_queries': 100},
 'per_query': {'002f5c970bac90e79f2b20e2d9e155d7619d6905': {'Recall@10': 0.5,
   'Precision@10': 0.2,
   'MRR@10': 1.0,
   'NDCG@10': 0.5205067333228022,
   'Recall@50': 0.5,
   'Precision@50': 0.04,
   'MRR@50': 1.0,
   'NDCG@50': 0.5205067333228022,
   'Recall@100': 0.5,
   'Precision@100': 0.02,
   'MRR@100': 1.0,
   'NDCG@100': 0.5205067333228022,
   'AP': 0.3214285714285714},
  '01ce421a014b560cd7b71029a22fd042990cda52': {'Recall@10': 0.42857142857142855,
   'Precision@10': 0.6,
   'MRR@10': 1.0,
   'NDCG@10': 0.7058075148678525,
  

## §2.6 — BM25L on Full Text

BM25L adds `delta` to the normalised term count before the saturation step, making
the length normalisation smoother than BM25+ for long documents.

In [50]:
if _is_cached('bm25l_ft'):
    sub_bm25l_ft, scores_bm25l_ft = _load_cache('bm25l_ft')
    print('BM25L FT: loaded from cache. ✓')
else:
    BM25L_GRID = {'k1': [1.5, 2.0, 2.5], 'b': [0.65, 0.75, 0.85, 0.9], 'delta': [0.25, 0.5, 0.75, 1.0]}
    print('Fitting BM25L (one vocab fit) ...')
    _bm25l = BM25L().fit(corpus_full)
    print(f'  vocab size: {len(_bm25l._vec.vocabulary_):,}\n')

    combos     = list(product(BM25L_GRID['k1'], BM25L_GRID['b'], BM25L_GRID['delta']))
    bm25l_rows = []
    for k1, b, delta in tqdm(combos, desc='BM25L grid'):
        _bm25l.k1 = k1; _bm25l.b = b; _bm25l.delta = delta; _bm25l._build_W()
        scores = _bm25l.score_matrix(query_texts)
        r100   = quick_recall100(scores, query_ids, corpus_ids, qrels)
        bm25l_rows.append((r100, k1, b, delta))

    bm25l_rows.sort(reverse=True)
    print(f"\n{'k1':>5} {'b':>5} {'delta':>6}  Recall@100"); print('-'*34)
    for r100, k1, b, delta in bm25l_rows[:10]:
        print(f'{k1:>5.1f} {b:>5.2f} {delta:>6.2f}  {r100:.4f}')

    best = bm25l_rows[0]
    _bm25l.k1 = best[1]; _bm25l.b = best[2]; _bm25l.delta = best[3]; _bm25l._build_W()
    scores_bm25l_ft = _bm25l.score_matrix(query_texts)
    sub_bm25l_ft    = make_submission(scores_bm25l_ft, query_ids, corpus_ids)
    print(f'\nBest BM25L: k1={best[1]}, b={best[2]}, delta={best[3]}  Recall@100={best[0]:.4f}')
    _save_cache('bm25l_ft', sub_bm25l_ft, scores_bm25l_ft)
    print('BM25L FT: saved. ✓')

evaluate(sub_bm25l_ft, qrels, ks=[10, 50, 100], query_domains=query_domains)

Fitting BM25L (one vocab fit) ...
  vocab size: 162,542



BM25L grid: 100%|██████████| 48/48 [00:36<00:00,  1.33it/s]



   k1     b  delta  Recall@100
----------------------------------
  2.5  0.90   0.25  0.7937
  2.5  0.85   0.25  0.7909
  2.5  0.90   0.50  0.7858
  2.5  0.75   0.25  0.7857
  2.0  0.90   0.25  0.7845
  2.5  0.85   0.50  0.7822
  2.0  0.85   0.25  0.7795
  2.5  0.65   0.25  0.7788
  2.5  0.75   0.50  0.7758
  2.5  0.90   0.75  0.7757

Best BM25L: k1=2.5, b=0.9, delta=0.25  Recall@100=0.7937
BM25L FT: saved. ✓

OVERALL RESULTS
Recall          @ 10: 0.4743  @ 50: 0.7289  @100: 0.7937
Precision       @ 10: 0.2430  @ 50: 0.0968  @100: 0.0564
MRR             @ 10: 0.6828  @ 50: 0.6897  @100: 0.6899
NDCG            @ 10: 0.5176  @ 50: 0.5785  @100: 0.5975
MAP             0.4233
Queries         100

--------------------------------------------------------------------
PER-DOMAIN  (Recall@100)
--------------------------------------------------------------------
  Domain                       Recall@100   NDCG@10    MAP
  Art                          0.5000       0.5205     0.3214
  Biology    

{'overall': {'Recall@10': 0.4742961672431158,
  'Precision@10': 0.243,
  'MRR@10': 0.6828333333333333,
  'NDCG@10': 0.5175718643742061,
  'Recall@50': 0.7289137542965668,
  'Precision@50': 0.0968,
  'MRR@50': 0.6896721801000606,
  'NDCG@50': 0.5785266652760473,
  'Recall@100': 0.7936525481791257,
  'Precision@100': 0.056400000000000006,
  'MRR@100': 0.6899315228835657,
  'NDCG@100': 0.5975071298596883,
  'MAP': 0.423309053254193,
  'num_queries': 100},
 'per_query': {'002f5c970bac90e79f2b20e2d9e155d7619d6905': {'Recall@10': 0.5,
   'Precision@10': 0.2,
   'MRR@10': 1.0,
   'NDCG@10': 0.5205067333228022,
   'Recall@50': 0.5,
   'Precision@50': 0.04,
   'MRR@50': 1.0,
   'NDCG@50': 0.5205067333228022,
   'Recall@100': 0.5,
   'Precision@100': 0.02,
   'MRR@100': 1.0,
   'NDCG@100': 0.5205067333228022,
   'AP': 0.3214285714285714},
  '01ce421a014b560cd7b71029a22fd042990cda52': {'Recall@10': 0.42857142857142855,
   'Precision@10': 0.6,
   'MRR@10': 1.0,
   'NDCG@10': 0.7116179486056782,
  

---
## §2.7 — Run all BM25 models with best parameters (no grid search)

Use this for held-out queries where gold labels are unavailable.
Fill `BEST_PARAMS` from the grid-search output printed above.

In [51]:
# ── Best parameters from grid search — update these from §2.1–§2.6 output ──
BEST_PARAMS = {
    'bm25_ta':     dict(k1=1.5,  b=0.85),
    'bm25_ft':     dict(k1=2.0,  b=0.85),
    'bm25f_ta':    dict(k1=2.0,  title_w=3.0,  title_b=0.3,
                        abstract_b=0.85),
    'bm25f_ft':    dict(k1=2.0,  title_w=5.0,  abstract_w=2.0,
                        title_b=0.5,  abstract_b=0.65, body_b=0.9),
    'bm25plus_ft': dict(k1=2.5,  b=0.9,  delta=0.5),
    'bm25l_ft':    dict(k1=2.5,  b=0.9,  delta=0.25)
}

# Internal name → cache key mapping
_BM25_CACHE_KEYS = {
    'BM25 (TA)':  'bm25_ta',
    'BM25 (FT)':  'bm25_ft',
    'BM25F (TA)': 'bm25f_ta',
    'BM25F (FT)': 'bm25f_ft',
    'BM25+ (FT)': 'bm25plus_ft',
    'BM25L (FT)': 'bm25l_ft',
}

def run_all_bm25_best_params(
    corpus_texts, corpus_full, title_texts, abstract_texts, body_texts,
    query_texts, query_ta, query_ids, corpus_ids,
    top_k=TOP_K, params=None,
):
    """Fit and score all BM25 variants using fixed best parameters.
    Saves each submission + score matrix to disk (respects CACHE_PREFIX).

    Returns a dict matching ALL_SUBMISSIONS:
        {'BM25 (TA)': sub, 'BM25 (FT)': sub, ...}
    """
    if params is None:
        params = BEST_PARAMS

    subs = {}

    def _fit_and_save(name, model, q_texts):
        cache_key = _BM25_CACHE_KEYS[name]
        if _is_cached(cache_key):
            sub, _ = _load_cache(cache_key)
            print(f'  {name}: loaded from cache. ✓')
        else:
            scores = model.score_matrix(q_texts)
            sub    = make_submission(scores, query_ids, corpus_ids, top_k)
            _save_cache(cache_key, sub, scores)
            print(f'  {name}: scored & saved. ✓')
        subs[name] = sub

    # BM25 TA
    p = params['bm25_ta']
    print(f"Fitting BM25 TA  (k1={p['k1']}, b={p['b']}) ...")
    _fit_and_save('BM25 (TA)', BM25(k1=p['k1'], b=p['b']).fit(corpus_texts), query_texts)

    # BM25 FT
    p = params['bm25_ft']
    print(f"Fitting BM25 FT  (k1={p['k1']}, b={p['b']}) ...")
    _fit_and_save('BM25 (FT)', BM25(k1=p['k1'], b=p['b']).fit(corpus_full), query_texts)

    # BM25F TA
    p = params['bm25f_ta']
    print(f"Fitting BM25F TA (k1={p['k1']}, title_w={p['title_w']}, "
          f"title_b={p['title_b']}, abstract_b={p['abstract_b']}) ...")
    _m = BM25F(k1=p['k1'],
               field_weights={'title': p['title_w'], 'abstract': 1.0},
               field_b={'title': p['title_b'], 'abstract': p['abstract_b']})
    _m.fit({'title': title_texts, 'abstract': abstract_texts})
    _fit_and_save('BM25F (TA)', _m, query_texts)

    # BM25F FT
    p = params['bm25f_ft']
    print(f"Fitting BM25F FT (k1={p['k1']}, title_w={p['title_w']}, abstract_w={p['abstract_w']}, "
          f"title_b={p['title_b']}, abstract_b={p['abstract_b']}, body_b={p['body_b']}) ...")
    _m = BM25F(k1=p['k1'],
               field_weights={'title': p['title_w'], 'abstract': p['abstract_w'], 'body': 1.0},
               field_b={'title': p['title_b'], 'abstract': p['abstract_b'], 'body': p['body_b']})
    _m.fit({'title': title_texts, 'abstract': abstract_texts, 'body': body_texts})
    _fit_and_save('BM25F (FT)', _m, query_texts)

    # BM25+ FT
    p = params['bm25plus_ft']
    print(f"Fitting BM25+ FT (k1={p['k1']}, b={p['b']}, delta={p['delta']}) ...")
    _fit_and_save('BM25+ (FT)', BM25Plus(k1=p['k1'], b=p['b'], delta=p['delta']).fit(corpus_full), query_texts)

    # BM25L FT
    p = params['bm25l_ft']
    print(f"Fitting BM25L FT (k1={p['k1']}, b={p['b']}, delta={p['delta']}) ...")
    _fit_and_save('BM25L (FT)', BM25L(k1=p['k1'], b=p['b'], delta=p['delta']).fit(corpus_full), query_texts)

    print('\nAll BM25 models scored.')
    return subs

In [ ]:
ALL_SUBMISSIONS_BM25 = run_all_bm25_best_params(
    corpus_texts, corpus_full, title_texts, abstract_texts, body_texts,
    query_texts, query_ta, query_ids, corpus_ids,
)

Fitting BM25 TA  (k1=1.5, b=0.85) ...
  BM25 (TA): scored & saved. ✓
Fitting BM25 FT  (k1=2.0, b=0.85) ...
  BM25 (FT): scored & saved. ✓
Fitting BM25F TA (k1=2.0, title_w=3.0, title_b=0.3, abstract_b=0.85) ...
  BM25F (TA): scored & saved. ✓
Fitting BM25F FT (k1=2.0, title_w=5.0, abstract_w=2.0, title_b=0.5, abstract_b=0.65, body_b=0.9) ...
  BM25F (FT): scored & saved. ✓
Fitting BM25+ FT (k1=2.5, b=0.9, delta=0.5) ...
  BM25+ (FT): scored & saved. ✓
Fitting BM25L FT (k1=2.5, b=0.9, delta=0.25) ...
  BM25L (FT): scored & saved. ✓

All BM25 models scored.


---
# §3.5 — SPECTER-Proximity Dense Retrieval

Load pre-computed `allenai/specter2_proximity` embeddings and score every
query against the corpus via cosine similarity.  Results are cached to disk
so re-running is instant.

**Embedding dir**: `../specter_prox_embed/`  
Files: `corpus_embeddings.npy`, `queries_embeddings.npy` + matching `*_ids.json`

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity as _cos_sim

SPECTER_DIR = Path('../specter_prox_embed')
_sp_cache_key = 'specter_prox'

# For held-out: update SPECTER_QUERY_EMB in cell 2 and clear cache
# (delete submissions/submission_specter_prox.json + scores_specter_prox.npy)

if _is_cached(_sp_cache_key):
    sub_specter_prox, scores_specter_prox = _load_cache(_sp_cache_key)
    print('SPECTER Prox: loaded from cache. ✓')
else:
    # ── Load embeddings ───────────────────────────────────────────────────
    print('Loading SPECTER-Proximity embeddings ...')
    sp_corpus_emb = np.load(SPECTER_DIR / 'corpus_embeddings.npy').astype(np.float32)
    sp_query_emb  = np.load(SPECTER_DIR / SPECTER_QUERY_EMB).astype(np.float32)
    _sp_qids_file = SPECTER_QUERY_EMB.replace('_embeddings.npy', '_ids.json')

    with open(SPECTER_DIR / 'corpus_ids.json') as _f: sp_corpus_ids = json.load(_f)
    with open(SPECTER_DIR / _sp_qids_file) as _f: sp_query_ids  = json.load(_f)

    print(f'  corpus : {sp_corpus_emb.shape}  |  queries : {sp_query_emb.shape}')

    # ── Align to the notebook's corpus_ids / query_ids order ─────────────
    # (the embedding files may be ordered differently)
    sp_c_idx = {cid: i for i, cid in enumerate(sp_corpus_ids)}
    sp_q_idx = {qid: i for i, qid in enumerate(sp_query_ids)}

    c_order  = [sp_c_idx[cid] for cid in corpus_ids]
    q_order  = [sp_q_idx[qid] for qid in query_ids]

    sp_corpus_emb = sp_corpus_emb[c_order]   # (20000, 768) aligned
    sp_query_emb  = sp_query_emb[q_order]    # (N_q, 768) aligned

    # ── L2-normalise → dot product == cosine similarity ──────────────────
    sp_corpus_emb /= np.linalg.norm(sp_corpus_emb, axis=1, keepdims=True).clip(min=1e-9)
    sp_query_emb  /= np.linalg.norm(sp_query_emb,  axis=1, keepdims=True).clip(min=1e-9)

    # ── Score matrix (N_q × 20000) in batches to avoid OOM ───────────────
    BATCH = 20
    scores_specter_prox = np.zeros((len(query_ids), len(corpus_ids)), dtype=np.float32)
    for start in tqdm(range(0, len(query_ids), BATCH), desc='SPECTER cosine'):
        end = min(start + BATCH, len(query_ids))
        scores_specter_prox[start:end] = sp_query_emb[start:end] @ sp_corpus_emb.T

    sub_specter_prox = make_submission(scores_specter_prox, query_ids, corpus_ids)
    _save_cache(_sp_cache_key, sub_specter_prox, scores_specter_prox)
    print(f'SPECTER Prox: scored & cached. ✓  shape={scores_specter_prox.shape}')

SPECTER Prox: loaded from cache. ✓


---
# §3 — Full Comparison

Side-by-side Recall@100 across all sparse models.  The best model overall and
per-domain feeds directly into §4 routing.

In [ ]:
ALL_SUBMISSIONS = {
    'TF-IDF uni (TA)':  sub_tfidf_uni_ta,
    'TF-IDF bi (TA)':   sub_tfidf_bi_ta,
    'TF-IDF tri (TA)':  sub_tfidf_tri_ta,
    'TF-IDF uni (FT)':  sub_tfidf_uni_ft,
    'TF-IDF bi (FT)':   sub_tfidf_bi_ft,
    'BM25 (TA)':        sub_bm25_ta,
    'BM25 (FT)':        sub_bm25_ft,
    'BM25F (TA)':       sub_bm25f_ta,
    'BM25F (FT)':       sub_bm25f_ft,
    'BM25+ (FT)':       sub_bm25p_ft,
    'BM25L (FT)':       sub_bm25l_ft,
    'SPECTER Prox':     sub_specter_prox,
}

if HAS_QRELS:
    _all_results = compare_submissions(
        ALL_SUBMISSIONS, qrels,
        ks=[10, 50, 100], query_domains=query_domains,
    )
else:
    _all_results = {name: {'overall': {}, 'per_domain': {}} for name in ALL_SUBMISSIONS}
    print('HAS_QRELS=False — skipping evaluation. ✓')


SUBMISSION COMPARISON
Metric                TF-IDF uni (TA)   TF-IDF bi (TA)  TF-IDF tri (TA)  TF-IDF uni (FT)   TF-IDF bi (FT)        BM25 (TA)        BM25 (FT)       BM25F (TA)       BM25F (FT)       BM25+ (FT)       BM25L (FT)     SPECTER Prox  Best
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Recall@10                      0.0000           0.0000           0.0000           0.0000           0.0000           0.3928           0.4714           0.4119           0.4433           0.4747           0.4767           0.0000  BM25L (FT)
Precision@10                   0.0000           0.0000           0.0000           0.0000           0.0000           0.2040           0.2430           0.2130           0.2330           0.2450           0.2450           0.0000  BM25+ (FT)
MRR@10                         0.0000    

### Recall@100 leaderboard

Sorted by Recall@100 — the primary metric for the reranker's input quality.

In [ ]:
if HAS_QRELS:
    pass
    # (leaderboard below only runs with qrels)
if HAS_QRELS:
    print(f'\n{"Model":<22}  {"Recall@10":>10}  {"Recall@50":>10}  {"Recall@100":>11}  {"NDCG@10":>8}')
    print('─' * 72)
    _rows = []
    for name, sub in ALL_SUBMISSIONS.items():
        r10  = float(np.mean([recall_at_k(sub.get(q,[]), set(qrels[q]), 10)  for q in qrels if qrels[q]]))
        r50  = float(np.mean([recall_at_k(sub.get(q,[]), set(qrels[q]), 50)  for q in qrels if qrels[q]]))
        r100 = float(np.mean([recall_at_k(sub.get(q,[]), set(qrels[q]), 100) for q in qrels if qrels[q]]))
        n10  = float(np.mean([ndcg_at_k(sub.get(q,[]),   set(qrels[q]), 10)  for q in qrels if qrels[q]]))
        _rows.append((r100, name, r10, r50, n10))
    
    _rows.sort(reverse=True)
    for r100, name, r10, r50, n10 in _rows:
        print(f'{name:<22}  {r10:>10.4f}  {r50:>10.4f}  {r100:>11.4f}  {n10:>8.4f}')
    
    print(f'\n  Best Recall@100: {_rows[0][1]}  ({_rows[0][0]:.4f})')



Model                    Recall@10   Recall@50   Recall@100   NDCG@10
────────────────────────────────────────────────────────────────────────
BM25L (FT)                  0.4767      0.7230       0.7935    0.5217
BM25 (FT)                   0.4714      0.7210       0.7889    0.5195
BM25+ (FT)                  0.4747      0.7158       0.7845    0.5175
BM25F (FT)                  0.4433      0.6701       0.7583    0.4924
BM25F (TA)                  0.4119      0.6020       0.6865    0.4509
BM25 (TA)                   0.3928      0.5934       0.6822    0.4265
TF-IDF uni (TA)             0.0000      0.0000       0.0000    0.0000
TF-IDF uni (FT)             0.0000      0.0000       0.0000    0.0000
TF-IDF tri (TA)             0.0000      0.0000       0.0000    0.0000
TF-IDF bi (TA)              0.0000      0.0000       0.0000    0.0000
TF-IDF bi (FT)              0.0000      0.0000       0.0000    0.0000
SPECTER Prox                0.0000      0.0000       0.0000    0.0000

  Best Recall@1

---
# §4 — Domain Routing

Different domains may be better served by different sparse models — e.g. a domain
with very long documents benefits from full-text BM25, while a domain with short
titles may prefer TA-only BM25F.

**Routing metric: Recall@100** — we pick the model with the highest Recall@100 on
each domain, then build a single routed submission that mixes models per query.

The per-domain winner table below drives the routing decision.

In [ ]:
if HAS_QRELS:
    _ROUTE_METRIC = 'Recall@100'
    _domains      = sorted(set(query_domains.values()))
    
    DOMAIN_CONFIG = {}
    DEFAULT_MODEL = max(ALL_SUBMISSIONS,
                        key=lambda n: _all_results[n]['overall'].get(_ROUTE_METRIC, 0.0))
    
    print(f'Routing criterion : {_ROUTE_METRIC}\n')
    print(f"{' ':<28} {'Best model':<22}  {_ROUTE_METRIC:>10}  {'Recall@50':>10}  {'Queries':>7}")
    print('─' * 82)
    for domain in _domains:
        vals     = {name: _all_results[name].get('per_domain', {}).get(domain, {}).get(_ROUTE_METRIC, 0.0)
                    for name in ALL_SUBMISSIONS}
        r50_vals = {name: _all_results[name].get('per_domain', {}).get(domain, {}).get('Recall@50', 0.0)
                    for name in ALL_SUBMISSIONS}
        best  = max(vals, key=vals.get)
        n_q   = sum(1 for d in query_domains.values() if d == domain)
        DOMAIN_CONFIG[domain] = best
        print(f'{domain:<28} {best:<22}  {vals[best]:>10.4f}  {r50_vals[best]:>10.4f}  {n_q:>7}')
    
    print(f'\nDefault model (best overall {_ROUTE_METRIC}): {DEFAULT_MODEL}')


Routing criterion : Recall@100

                             Best model              Recall@100   Recall@50  Queries
──────────────────────────────────────────────────────────────────────────────────
Biology                      TF-IDF uni (TA)             0.0000      0.0000       20
Business                     TF-IDF uni (TA)             0.0000      0.0000        2
Chemistry                    TF-IDF uni (TA)             0.0000      0.0000       10
Computer Science             TF-IDF uni (TA)             0.0000      0.0000       18
Economics                    TF-IDF uni (TA)             0.0000      0.0000        1
Engineering                  TF-IDF uni (TA)             0.0000      0.0000        1
Environmental Science        TF-IDF uni (TA)             0.0000      0.0000        3
Geography                    TF-IDF uni (TA)             0.0000      0.0000        2
Geology                      TF-IDF uni (TA)             0.0000      0.0000        1
Materials Science            TF-IDF

In [ ]:
# ── Manual domain routing (for heldout queries without qrels) ───────────────
# When running on heldout data you have no ground truth to auto-select the best
# model per domain.  Copy the winning assignments printed by the cell above and
# set them here.  Leave MANUAL_ROUTING = False to use the auto-selected config.

# When HAS_QRELS=False there is no ground truth to auto-select — force manual routing.
MANUAL_ROUTING = True if not HAS_QRELS else False  # ← override to True for held-out

if MANUAL_ROUTING:
    # Keys  : domain strings (run `sorted(set(queries['domain']))` to list them)
    # Values: model name — must be a key in ALL_SUBMISSIONS
    DOMAIN_CONFIG = {
        "Art": "TF-IDF uni (TA)",
        "Biology": "TF-IDF uni (FT)",
        "Business": "TF-IDF bi (TA)",
        "Chemistry": "TF-IDF uni (FT)",
        "Computer Science": "TF-IDF uni (FT)",
        "Economics": "SPECTER Prox",
        "Engineering": "TF-IDF uni (TA)",
        "Environmental Science": "TF-IDF uni (TA)",
        "Geography": "TF-IDF bi (TA)",
        "Geology": "TF-IDF uni (TA)",
        "History": "TF-IDF uni (TA)",
        "Materials Science": "TF-IDF uni (FT)",
        "Mathematics": "TF-IDF uni (TA)",
        "Medicine": "TF-IDF uni (FT)",
        "Philosophy": "TF-IDF uni (FT)",
        "Physics":  "TF-IDF bi (FT)",
        "Political Science": "TF-IDF uni (FT)",
        "Psychology": "TF-IDF uni (TA)",
        "Sociology": "TF-IDF bi (FT)"
    }
    DEFAULT_MODEL = 'TF-IDF uni (FT)'  # ← model used for any domain not listed above
    print('Manual routing active.')
    print(f'  DEFAULT_MODEL : {DEFAULT_MODEL}')
    print(f'  DOMAIN_CONFIG : {DOMAIN_CONFIG}')
else:
    print('Auto routing active (DOMAIN_CONFIG set by the cell above).')

# Sanity check — all values must be valid model names
unknown = [m for m in DOMAIN_CONFIG.values() if m not in ALL_SUBMISSIONS]
if unknown:
    raise ValueError(f'Unknown model(s) in DOMAIN_CONFIG: {unknown}\n'
                     f'Valid names: {list(ALL_SUBMISSIONS)}')
print('DOMAIN_CONFIG OK ✓')

Auto routing active (DOMAIN_CONFIG set by the cell above).
DOMAIN_CONFIG OK ✓


## §4.1 — Build routed submission

For each query we look up its domain and select the model assigned in `DOMAIN_CONFIG`.
The resulting `sub_routed` is what feeds Phase 1 of `hybrid_pipeline.ipynb`.

In [ ]:
_route_cache = SUBMISSIONS_DIR / f'submission_sparse_routed_top{TOP_K}.json'

if _route_cache.exists():
    with open(_route_cache) as _f: sub_routed = json.load(_f)
    print(f'Sparse routed (top-{TOP_K}): loaded from cache. ✓')
else:
    from collections import Counter
    sub_routed  = {}
    _model_used = {}
    for qid in tqdm(query_ids, desc='Building routed submission'):
        domain = query_domains.get(qid, '')
        model  = DOMAIN_CONFIG.get(domain, DEFAULT_MODEL)
        sub_routed[qid]  = ALL_SUBMISSIONS[model].get(qid, [])[:TOP_K]
        _model_used[qid] = model
    with open(_route_cache, 'w') as _f: json.dump(sub_routed, _f)
    print(f'Sparse routed (top-{TOP_K}): built and saved. ✓')

    print('\nModel usage:')
    for m, c in Counter(_model_used.values()).most_common():
        print(f'  {m:<22}  {c:>5} queries')

compare_submissions(
    {DEFAULT_MODEL:              ALL_SUBMISSIONS[DEFAULT_MODEL],
     f'Routed (top-{TOP_K})':    sub_routed},
    qrels, ks=[10, 50, 100], query_domains=query_domains,
)

Building routed submission: 100%|██████████| 100/100 [00:00<00:00, 60021.52it/s]

Sparse routed (top-100): built and saved. ✓

Model usage:
  TF-IDF uni (TA)           100 queries

SUBMISSION COMPARISON
Metric                      BM25L (FT)  Routed (top-100)  Best     Delta
------------------------------------------------------------------------
Recall@10                       0.4767            0.0000  BM25L (FT)   -0.4767
Precision@10                    0.2450            0.0000  BM25L (FT)   -0.2450
MRR@10                          0.6881            0.0000  BM25L (FT)   -0.6881
NDCG@10                         0.5217            0.0000  BM25L (FT)   -0.5217
Recall@50                       0.7230            0.0000  BM25L (FT)   -0.7230
Precision@50                    0.0970            0.0000  BM25L (FT)   -0.0970
MRR@50                          0.6945            0.0000  BM25L (FT)   -0.6945
NDCG@50                         0.5792            0.0000  BM25L (FT)   -0.5792
Recall@100                      0.7935            0.0000  BM25L (FT)   -0.7935
Precision@100         

{'BM25L (FT)': {'overall': {'Recall@10': 0.47670077671814776,
   'Precision@10': 0.24500000000000002,
   'MRR@10': 0.688095238095238,
   'NDCG@10': 0.5216615626627511,
   'Recall@50': 0.7230237338165962,
   'Precision@50': 0.09699999999999999,
   'MRR@50': 0.6945379611594548,
   'NDCG@50': 0.5792112732414652,
   'Recall@100': 0.7934799751917946,
   'Precision@100': 0.056100000000000004,
   'MRR@100': 0.6947792402292222,
   'NDCG@100': 0.5986078135381672,
   'MAP': 0.42583993336924864,
   'num_queries': 100},
  'per_query': {'002f5c970bac90e79f2b20e2d9e155d7619d6905': {'Recall@10': 0.5,
    'Precision@10': 0.2,
    'MRR@10': 1.0,
    'NDCG@10': 0.5135312443624667,
    'Recall@50': 0.5,
    'Precision@50': 0.04,
    'MRR@50': 1.0,
    'NDCG@50': 0.5135312443624667,
    'Recall@100': 0.5,
    'Precision@100': 0.02,
    'MRR@100': 1.0,
    'NDCG@100': 0.5135312443624667,
    'AP': 0.3125},
   '01ce421a014b560cd7b71029a22fd042990cda52': {'Recall@10': 0.42857142857142855,
    'Precision@10':

---
# §5 — Recall@k ceiling summary

The table below shows Recall at multiple cut-offs for every model and the routed
ensemble.  The `Recall@100` column for the routed row is the recall ceiling that
`hybrid_pipeline.ipynb` starts from — anything above this can only come from the
dense retrieval stage.

In [ ]:
if HAS_QRELS:
    _eval_subs = {**ALL_SUBMISSIONS, f'★ Routed (top-{TOP_K})': sub_routed}
    _ks        = [10, 50, 100]
    
    print(f'\n{"Model":<28}' + ''.join(f'  {"R@"+str(k):>9}' for k in _ks))
    print('─' * (28 + 11 * len(_ks)))
    
    _summary_rows = []
    for name, sub in _eval_subs.items():
        vals = [
            float(np.mean([recall_at_k(sub.get(q,[]), set(qrels[q]), k) for q in qrels if qrels[q]]))
            for k in _ks
        ]
        _summary_rows.append((vals[-1], name, vals))
    
    _summary_rows.sort(reverse=True)
    for r100, name, vals in _summary_rows:
        print(f'{name:<28}' + ''.join(f'  {v:>9.4f}' for v in vals))
    
    print(f'\n  Recall@100 ceiling going into hybrid_pipeline.ipynb:  '
          f'{max(vals[-1] for _, _, vals in _summary_rows):.4f}')



Model                              R@10       R@50      R@100
─────────────────────────────────────────────────────────────
BM25L (FT)                       0.4767     0.7230     0.7935
BM25 (FT)                        0.4714     0.7210     0.7889
BM25+ (FT)                       0.4747     0.7158     0.7845
BM25F (FT)                       0.4433     0.6701     0.7583
BM25F (TA)                       0.4119     0.6020     0.6865
BM25 (TA)                        0.3928     0.5934     0.6822
★ Routed (top-100)               0.0000     0.0000     0.0000
TF-IDF uni (TA)                  0.0000     0.0000     0.0000
TF-IDF uni (FT)                  0.0000     0.0000     0.0000
TF-IDF tri (TA)                  0.0000     0.0000     0.0000
TF-IDF bi (TA)                   0.0000     0.0000     0.0000
TF-IDF bi (FT)                   0.0000     0.0000     0.0000
SPECTER Prox                     0.0000     0.0000     0.0000

  Recall@100 ceiling going into hybrid_pipeline.ipynb:  0.7935


---
# §6 — Hybrid Domain Routing (Sparse + SPECTER-Prox via RRF)

For each query we fuse the domain-best sparse model with SPECTER-Proximity
using **Reciprocal Rank Fusion (RRF)**.  We then re-run domain routing on
the fused scores, producing a hybrid routed submission.

RRF formula: `score(d) = 1/(k + rank_sparse(d)) + 1/(k + rank_dense(d))`  with `k=60`.

In [ ]:
# ── RRF helper ────────────────────────────────────────────────────────────────
def rrf_fuse(score_matrix_a, score_matrix_b, k=60):
    """Reciprocal Rank Fusion of two (N_q, N_c) score matrices."""
    n_q, n_c = score_matrix_a.shape
    fused = np.zeros((n_q, n_c), dtype=np.float32)
    for i in range(n_q):
        ranks_a = np.argsort(np.argsort(-score_matrix_a[i])) + 1  # 1-based rank
        ranks_b = np.argsort(np.argsort(-score_matrix_b[i])) + 1
        fused[i] = 1.0 / (k + ranks_a) + 1.0 / (k + ranks_b)
    return fused

# ── Build one hybrid submission per sparse model fused with SPECTER Prox ─────
_SPARSE_SCORE_MAP = {
    'TF-IDF uni (TA)': 'tfidf_uni_ta',
    'TF-IDF bi (TA)':  'tfidf_bi_ta',
    'TF-IDF tri (TA)': 'tfidf_tri_ta',
    'TF-IDF uni (FT)': 'tfidf_uni_ft',
    'TF-IDF bi (FT)':  'tfidf_bi_ft',
    'BM25 (TA)':       'bm25_ta',
    'BM25 (FT)':       'bm25_ft',
    'BM25F (TA)':      'bm25f_ta',
    'BM25F (FT)':      'bm25f_ft',
    'BM25+ (FT)':      'bm25plus_ft',
    'BM25L (FT)':      'bm25l_ft',
}

HYBRID_SUBMISSIONS = {}
HYBRID_SCORES      = {}

for sparse_name, cache_key in tqdm(_SPARSE_SCORE_MAP.items(), desc='RRF fuse'):
    hkey = f'hybrid_{cache_key}'
    if _is_cached(hkey):
        HYBRID_SUBMISSIONS[sparse_name], HYBRID_SCORES[sparse_name] = _load_cache(hkey)
    else:
        _, sparse_scores = _load_cache(cache_key)
        fused = rrf_fuse(sparse_scores, scores_specter_prox)
        sub   = make_submission(fused, query_ids, corpus_ids)
        _save_cache(hkey, sub, fused)
        HYBRID_SUBMISSIONS[sparse_name] = sub
        HYBRID_SCORES[sparse_name]      = fused

print(f'Built {len(HYBRID_SUBMISSIONS)} hybrid submissions.')

RRF fuse: 100%|██████████| 11/11 [00:10<00:00,  1.10it/s]

Built 11 hybrid submissions.


In [ ]:
# Fixed HYBRID_DOMAIN_CONFIG for held-out (copy winning assignments from training run)
MANUAL_HYBRID_ROUTING = not HAS_QRELS

if not MANUAL_HYBRID_ROUTING:
    # ── Evaluate all hybrid submissions ──────────────────────────────────────────
    _hybrid_results = compare_submissions(
        HYBRID_SUBMISSIONS, qrels,
        ks=[10, 50, 100], query_domains=query_domains,
    )
    
    # ── Domain routing on hybrid submissions ─────────────────────────────────────
    _HYBRID_ROUTE_METRIC = 'Recall@100'
    
    HYBRID_DOMAIN_CONFIG = {}
    HYBRID_DEFAULT_MODEL = max(
        HYBRID_SUBMISSIONS,
        key=lambda n: _hybrid_results[n]['overall'].get(_HYBRID_ROUTE_METRIC, 0.0)
    )
    
    print(f'Hybrid routing criterion : {_HYBRID_ROUTE_METRIC}\n')
    print(f"{' ':<28} {'Best hybrid model':<22}  {_HYBRID_ROUTE_METRIC:>10}  {'Recall@50':>10}  {'Queries':>7}")
    print('─' * 90)
    
    for domain in _domains:
        vals = {name: _hybrid_results[name].get('per_domain', {}).get(domain, {}).get(_HYBRID_ROUTE_METRIC, 0.0)
                for name in HYBRID_SUBMISSIONS}
        r50  = {name: _hybrid_results[name].get('per_domain', {}).get(domain, {}).get('Recall@50', 0.0)
                for name in HYBRID_SUBMISSIONS}
        best = max(vals, key=vals.get)
        n_q  = sum(1 for d in query_domains.values() if d == domain)
        HYBRID_DOMAIN_CONFIG[domain] = best
        print(f'{domain:<28} {best:<22}  {vals[best]:>10.4f}  {r50[best]:>10.4f}  {n_q:>7}')
    
    print(f'\nDefault hybrid model (best overall): {HYBRID_DEFAULT_MODEL}')
else:
    # Use the routing learned during training — copy from cell 44 training output
    HYBRID_DOMAIN_CONFIG = dict(DOMAIN_CONFIG)  # start from sparse routing
    HYBRID_DEFAULT_MODEL = max(
        HYBRID_SUBMISSIONS,
        key=lambda n: sum(1 for _ in HYBRID_SUBMISSIONS[n]),  # fallback: largest list
    )
    print("Manual hybrid routing active — using sparse DOMAIN_CONFIG as fallback. ✓")
    print("Tip: hardcode HYBRID_DOMAIN_CONFIG here from the training run output.")



SUBMISSION COMPARISON
Metric                TF-IDF uni (TA)   TF-IDF bi (TA)  TF-IDF tri (TA)  TF-IDF uni (FT)   TF-IDF bi (FT)        BM25 (TA)        BM25 (FT)       BM25F (TA)       BM25F (FT)       BM25+ (FT)       BM25L (FT)  Best
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Recall@10                      0.0000           0.0000           0.0000           0.0000           0.0000           0.0000           0.0000           0.0000           0.0000           0.0000           0.0000  TF-IDF uni (TA)
Precision@10                   0.0000           0.0000           0.0000           0.0000           0.0000           0.0000           0.0000           0.0000           0.0000           0.0000           0.0000  TF-IDF uni (TA)
MRR@10                         0.0000           0.0000           0.0000           0.0000           

In [ ]:
# ── Build routed hybrid submission ────────────────────────────────────────────
_hybrid_route_cache = SUBMISSIONS_DIR / f'submission_hybrid_routed_top{TOP_K}.json'

if _hybrid_route_cache.exists():
    with open(_hybrid_route_cache) as _f: sub_hybrid_routed = json.load(_f)
    print(f'Hybrid routed (top-{TOP_K}): loaded from cache. ✓')
else:
    from collections import Counter as _Counter
    sub_hybrid_routed  = {}
    _hybrid_model_used = {}
    for qid in tqdm(query_ids, desc='Building hybrid routed submission'):
        domain = query_domains.get(qid, '')
        model  = HYBRID_DOMAIN_CONFIG.get(domain, HYBRID_DEFAULT_MODEL)
        sub_hybrid_routed[qid]  = HYBRID_SUBMISSIONS[model].get(qid, [])[:TOP_K]
        _hybrid_model_used[qid] = model
    with open(_hybrid_route_cache, 'w') as _f: json.dump(sub_hybrid_routed, _f)
    print(f'Hybrid routed (top-{TOP_K}): built and saved. ✓')

    print('\nModel usage:')
    for m, c in _Counter(_hybrid_model_used.values()).most_common():
        print(f'  {m:<22}  {c:>5} queries')

# ── Final comparison: sparse routed vs hybrid routed ─────────────────────────
compare_submissions(
    {f'★ Sparse Routed (top-{TOP_K})': sub_routed,
     f'★ Hybrid Routed (top-{TOP_K})': sub_hybrid_routed},
    qrels, ks=[10, 50, 100], query_domains=query_domains,
)

Building hybrid routed submission: 100%|██████████| 100/100 [00:00<00:00, 225258.00it/s]

Hybrid routed (top-100): built and saved. ✓

Model usage:
  TF-IDF uni (TA)           100 queries

SUBMISSION COMPARISON
Metric                ★ Sparse Routed (top-100)  ★ Hybrid Routed (top-100)  Best     Delta
------------------------------------------------------------------------------------------
Recall@10                                0.0000                     0.0000  ★ Sparse Routed (top-100)   +0.0000
Precision@10                             0.0000                     0.0000  ★ Sparse Routed (top-100)   +0.0000
MRR@10                                   0.0000                     0.0000  ★ Sparse Routed (top-100)   +0.0000
NDCG@10                                  0.0000                     0.0000  ★ Sparse Routed (top-100)   +0.0000
Recall@50                                0.0000                     0.0000  ★ Sparse Routed (top-100)   +0.0000
Precision@50                             0.0000                     0.0000  ★ Sparse Routed (top-100)   +0.0000
MRR@50                   

{'★ Sparse Routed (top-100)': {'overall': {'Recall@10': 0.0,
   'Precision@10': 0.0,
   'MRR@10': 0.0,
   'NDCG@10': 0.0,
   'Recall@50': 0.0,
   'Precision@50': 0.0,
   'MRR@50': 0.0,
   'NDCG@50': 0.0,
   'Recall@100': 0.0,
   'Precision@100': 0.0,
   'MRR@100': 0.0,
   'NDCG@100': 0.0,
   'MAP': 0.0,
   'num_queries': 100},
  'per_query': {'002f5c970bac90e79f2b20e2d9e155d7619d6905': {'Recall@10': 0.0,
    'Precision@10': 0.0,
    'MRR@10': 0.0,
    'NDCG@10': 0.0,
    'Recall@50': 0.0,
    'Precision@50': 0.0,
    'MRR@50': 0.0,
    'NDCG@50': 0.0,
    'Recall@100': 0.0,
    'Precision@100': 0.0,
    'MRR@100': 0.0,
    'NDCG@100': 0.0,
    'AP': 0.0},
   '01ce421a014b560cd7b71029a22fd042990cda52': {'Recall@10': 0.0,
    'Precision@10': 0.0,
    'MRR@10': 0.0,
    'NDCG@10': 0.0,
    'Recall@50': 0.0,
    'Precision@50': 0.0,
    'MRR@50': 0.0,
    'NDCG@50': 0.0,
    'Recall@100': 0.0,
    'Precision@100': 0.0,
    'MRR@100': 0.0,
    'NDCG@100': 0.0,
    'AP': 0.0},
   '021ff5083b

---
# §7 — Domain-Confusion Reranking (NDCG@10 booster)

The confusion matrix tells us: *for a query in domain X, what fraction of
relevant documents come from each corpus domain?*

We use those weights directly to push high-weight documents to the top-10:

1. Each candidate gets a **domain weight** = `confusion_matrix[query_domain][doc_domain]`
2. Candidates are sorted by **(domain_weight ↓, original_rank ↑)**
   - Primary-domain docs (weight ≈ 1.0) come first
   - Related-domain docs (small weight > 0) come next
   - Unrelated docs (weight = 0) come last
3. Within each weight tier, the original retrieval order is preserved


In [ ]:
# ── Load the domain confusion matrix ─────────────────────────────────────────
import pandas as pd

confusion_df = pd.read_csv(Path('../domain_confusion_matrix_normalized.xls'), index_col=0)

# corpus doc_id → domain
corpus_domain_map = dict(zip(corpus['doc_id'], corpus['domain']))

# {query_domain: {doc_domain: weight}}
domain_weights = {qd: confusion_df.loc[qd].to_dict() for qd in confusion_df.index}

print('Confusion matrix loaded:', confusion_df.shape)
print('Non-diagonal weights (cross-domain relevance):')
for qd, row in domain_weights.items():
    cross = {dd: round(w, 4) for dd, w in row.items() if dd != qd and w > 0}
    if cross:
        print(f'  {qd:<25} → {cross}')


Confusion matrix loaded: (19, 19)
Non-diagonal weights (cross-domain relevance):
  Biology                   → {'Chemistry': 0.0157, 'Medicine': 0.0157}
  Business                  → {'Economics': 0.0556, 'Mathematics': 0.0556}
  Chemistry                 → {'Engineering': 0.0083, 'Materials Science': 0.0083, 'Mathematics': 0.0083, 'Medicine': 0.0167}
  Computer Science          → {'Engineering': 0.0065}
  Economics                 → {'Medicine': 0.0588}
  Environmental Science     → {'Geology': 0.0476}
  Geography                 → {'Medicine': 0.1111}
  Materials Science         → {'Computer Science': 0.0169, 'Physics': 0.0339}


In [ ]:
# ── Domain-weight reranker ────────────────────────────────────────────────────
# Sort by (domain_weight desc, original_rank asc).
# Same-domain docs come first; within a tier, original retrieval order is kept.
# Note: using SPECTER similarity as secondary sort was tested and found to hurt
# performance (0.63 vs 0.69), so we preserve the original retrieval rank instead.

def domain_rerank(submission, q_domains, corp_domain_map, dom_weights, top_k=TOP_K):
    reranked = {}
    for qid, cands in submission.items():
        q_domain = q_domains.get(qid, "")
        w_row    = dom_weights.get(q_domain, {})
        scored   = [
            (w_row.get(corp_domain_map.get(doc, ""), 0.0), rank, doc)
            for rank, doc in enumerate(cands)
        ]
        scored.sort(key=lambda x: (-x[0], x[1]))
        reranked[qid] = [doc for _, _, doc in scored[:top_k]]
    return reranked

# Apply to both routed submissions
sub_sparse_domain_reranked = domain_rerank(
    sub_routed,        query_domains, corpus_domain_map, domain_weights)
sub_hybrid_domain_reranked = domain_rerank(
    sub_hybrid_routed, query_domains, corpus_domain_map, domain_weights)

print("Domain reranking done. ✓")


Domain reranking done. ✓


In [ ]:
# ── Save & evaluate ───────────────────────────────────────────────────────────
with open(SUBMISSIONS_DIR / f'submission_sparse_domain_reranked_top{TOP_K}.json', 'w') as f:
    json.dump(sub_sparse_domain_reranked, f)
with open(SUBMISSIONS_DIR / f'submission_hybrid_domain_reranked_top{TOP_K}.json', 'w') as f:
    json.dump(sub_hybrid_domain_reranked, f)
print('Submissions saved. ✓')

if HAS_QRELS:
    compare_submissions(
        {'Sparse Routed':           sub_routed,
     'Hybrid Routed':           sub_hybrid_routed,
     'Sparse + Domain Rerank':  sub_sparse_domain_reranked,
     'Hybrid + Domain Rerank':  sub_hybrid_domain_reranked},
    qrels, ks=[10, 50, 100], query_domains=query_domains,
    )


Submissions saved. ✓

SUBMISSION COMPARISON
Metric                         Sparse Routed           Hybrid Routed  Sparse + Domain Rerank  Hybrid + Domain Rerank  Best
--------------------------------------------------------------------------------------------------------------------------
Recall@10                             0.0000                  0.0000                  0.0000                  0.0000  Sparse Routed
Precision@10                          0.0000                  0.0000                  0.0000                  0.0000  Sparse Routed
MRR@10                                0.0000                  0.0000                  0.0000                  0.0000  Sparse Routed
NDCG@10                               0.0000                  0.0000                  0.0000                  0.0000  Sparse Routed
Recall@50                             0.0000                  0.0000                  0.0000                  0.0000  Sparse Routed
Precision@50                          0.0000      

---
# §8 — 4-way RRF + Domain Reranking (best pipeline)

Fuse **four ranked lists** with RRF k=3, then apply domain-confusion reranking.
This combination was found by sweeping lists and k values against NDCG@10.

| List | Source |
|------|--------|
| `sub_sparse_routed` | Domain-routed BM25/TF-IDF |
| `sub_bge` | BGE-large bi-encoder (from `hybrid_pipeline.ipynb`) |
| `sub_specter_prox` | SPECTER2-Proximity bi-encoder |
| `sub_bm25f_ft` | BM25F on full text (single best BM25 variant) |

Result vs baseline:
- Baseline `hybrid_domain_reranked`: **NDCG@10 = 0.685**
- **4-way RRF k=3 + domain rerank: NDCG@10 = 0.727**


In [ ]:
# ── Load the four input lists ─────────────────────────────────────────────────
# sub_routed and sub_specter_prox are already in memory from §3.5 / §4.
# BGE dense submission comes from hybrid_pipeline.ipynb.

_bge_path    = SUBMISSIONS_DIR / 'submission_bge_dense_top100.json'
_bm25f_path  = SUBMISSIONS_DIR / 'submission_sparse_routed_top100.json'

if not _bge_path.exists():
    raise FileNotFoundError('Run hybrid_pipeline.ipynb Phase 2 first to generate BGE embeddings.')

with open(_bge_path)   as _f: sub_bge_dense = json.load(_f)
with open(_bm25f_path) as _f: sub_bm25f_ft  = json.load(_f)

print('All four lists loaded. ✓')

# ── 4-way RRF ─────────────────────────────────────────────────────────────────
RRF_K = 3   # tuned: k=3 maximises NDCG@10 on this dataset

_4rrf_cache = SUBMISSIONS_DIR / f'submission_4rrf_top{TOP_K}.json'

if _4rrf_cache.exists():
    with open(_4rrf_cache) as _f: sub_4rrf = json.load(_f)
    print(f'4-way RRF: loaded from cache. ✓')
else:
    _four_lists = [sub_routed, sub_bge_dense, sub_specter_prox, sub_bm25f_ft]
    sub_4rrf = {}
    for qid in tqdm(query_ids, desc='4-way RRF'):
        sc = {}
        for lst in _four_lists:
            for rank, doc in enumerate(lst.get(qid, []), 1):
                sc[doc] = sc.get(doc, 0.0) + 1.0 / (RRF_K + rank)
        sub_4rrf[qid] = sorted(sc, key=sc.get, reverse=True)[:TOP_K]
    with open(_4rrf_cache, 'w') as _f: json.dump(sub_4rrf, _f)
    print(f'4-way RRF: built and saved. ✓')

# ── Domain reranking on top of 4-way RRF ──────────────────────────────────────
sub_4rrf_domain = domain_rerank(sub_4rrf, query_domains, corpus_domain_map, domain_weights)

_out = SUBMISSIONS_DIR / f'submission_4rrf_domain_top{TOP_K}.json'
with open(_out, 'w') as _f: json.dump(sub_4rrf_domain, _f)
print(f'Saved: {_out.name}')

# ── Final comparison ──────────────────────────────────────────────────────────
compare_submissions(
    {'Hybrid domain reranked (baseline)': sub_hybrid_domain_reranked,
     '4-way RRF':                         sub_4rrf,
     '4-way RRF + domain rerank':          sub_4rrf_domain},
    qrels, ks=[10, 50, 100], query_domains=query_domains,
)


All four lists loaded. ✓
4-way RRF: loaded from cache. ✓
Saved: submission_4rrf_domain_top100.json

SUBMISSION COMPARISON
Metric                Hybrid domain reranked (baseline)                          4-way RRF          4-way RRF + domain rerank  Best
-----------------------------------------------------------------------------------------------------------------------------------
Recall@10                                        0.0000                             0.0000                             0.0000  Hybrid domain reranked (baseline)
Precision@10                                     0.0000                             0.0000                             0.0000  Hybrid domain reranked (baseline)
MRR@10                                           0.0000                             0.0000                             0.0000  Hybrid domain reranked (baseline)
NDCG@10                                          0.0000                             0.0000                             0.0000  Hybr

{'Hybrid domain reranked (baseline)': {'overall': {'Recall@10': 0.0,
   'Precision@10': 0.0,
   'MRR@10': 0.0,
   'NDCG@10': 0.0,
   'Recall@50': 0.0,
   'Precision@50': 0.0,
   'MRR@50': 0.0,
   'NDCG@50': 0.0,
   'Recall@100': 0.0,
   'Precision@100': 0.0,
   'MRR@100': 0.0,
   'NDCG@100': 0.0,
   'MAP': 0.0,
   'num_queries': 100},
  'per_query': {'002f5c970bac90e79f2b20e2d9e155d7619d6905': {'Recall@10': 0.0,
    'Precision@10': 0.0,
    'MRR@10': 0.0,
    'NDCG@10': 0.0,
    'Recall@50': 0.0,
    'Precision@50': 0.0,
    'MRR@50': 0.0,
    'NDCG@50': 0.0,
    'Recall@100': 0.0,
    'Precision@100': 0.0,
    'MRR@100': 0.0,
    'NDCG@100': 0.0,
    'AP': 0.0},
   '01ce421a014b560cd7b71029a22fd042990cda52': {'Recall@10': 0.0,
    'Precision@10': 0.0,
    'MRR@10': 0.0,
    'NDCG@10': 0.0,
    'Recall@50': 0.0,
    'Precision@50': 0.0,
    'MRR@50': 0.0,
    'NDCG@50': 0.0,
    'Recall@100': 0.0,
    'Precision@100': 0.0,
    'MRR@100': 0.0,
    'NDCG@100': 0.0,
    'AP': 0.0},
   '02

===================================================================================================================================
SUBMISSION COMPARISON
===================================================================================================================================
Metric                Hybrid domain reranked (baseline)                          4-way RRF          4-way RRF + domain rerank  Best
-----------------------------------------------------------------------------------------------------------------------------------
Recall@10                                        0.6550                             0.5958                             0.6844  4-way RRF + domain rerank
Precision@10                                     0.3350                             0.3000                             0.3390  4-way RRF + domain rerank
MRR@10                                           0.7861                             0.7494                             0.8472  4-way RRF + domain rerank
NDCG@10                                          0.6850                             0.6075                             0.7272  4-way RRF + domain rerank


---
# §9 — Held-out Inference Checklist

Run on new queries with **no qrels**. Follow these steps in order.

## Step 1 — Set flags in cell 2 (imports)
```python
HAS_QRELS         = False
QUERIES_PATH      = DATA_DIR / 'queries_2.parquet'   # ← your held-out queries
SPECTER_QUERY_EMB = 'queries_2_embeddings.npy'       # ← SPECTER embeddings for held-out queries
CACHE_PREFIX      = 'heldout_'                       # ← prevents overwriting training caches
```

## Step 2 — Prepare SPECTER query embeddings
Run `specter_embeddings_kaggle.ipynb` on the held-out queries.  
Copy `queries_embeddings.npy` and `queries_ids.json` to `../specter_prox_embed/`  
and rename them to match `SPECTER_QUERY_EMB` above.

## Step 3 — Prepare BGE query embeddings (for 4-way RRF)
In `hybrid_pipeline.ipynb` change `QUERIES_PATH` to the held-out queries parquet  
and run Phase 2 (BGE embedding). This writes `bge_large_query_emb.npy` to `../submissions/`.

## Step 4 — Run the notebook cells in this order
| Cell | What it does | Safe for held-out? |
|------|-------------|-------------------|
| §1 imports | loads libraries | ✓ |
| §2 helpers | defines functions | ✓ |
| Data loading | loads new queries + corpus (no qrels) | ✓ |
| §1 TF-IDF | loads from cache (corpus unchanged) | ✓ |
| §2 BM25 variants | loads from cache (corpus unchanged) | ✓ |
| §2.7 BM25 best params | scores **new queries** using BEST_PARAMS | ✓ |
| §3 Full Comparison | **skipped** (no qrels) | ✓ guarded |
| §4 Auto-routing | **skipped** (no qrels) | ✓ guarded |
| §4 Manual routing | runs — uses pre-set DOMAIN_CONFIG | ✓ |
| §4.1 Build routed | builds sub_routed | ✓ |
| §5 Ceiling summary | **skipped** (no qrels) | ✓ guarded |
| §3.5 SPECTER | computes scores for held-out queries | ✓ |
| §6 Hybrid RRF+domain | builds HYBRID_SUBMISSIONS | ✓ |
| §6 Hybrid routing | uses DOMAIN_CONFIG fallback | ✓ guarded |
| §7 Confusion rerank | applies domain weights | ✓ |
| §8 4-way RRF | fuses all four lists | ✓ |

## Step 5 — Output
Final submission: `../submissions/heldout_submission_4rrf_domain_top100.json`

## ⚠ Cache warning
Setting `CACHE_PREFIX = 'heldout_'` ensures held-out results are saved as  
`submission_heldout_*.json` and do **not** overwrite your training caches.  
If you forget this, delete the affected `.json` / `.npy` files before re-running training.
